## 0. Imports

In [ ]:
# import os 
# os.environ['CUDA_VISIBLE_DEVICES'] = '1'
import sys
sys.path.append("..")

from Benchmark_Models.NAS_Models import NAS_TopModels_Runner
from Benchmark_Models.NAS_Chain_Models import NAS_TopModels_Runner as NAS_Chain_Runner
from Benchmark_Models.extract_NAS_trees import _episode_indices
from Benchmark_Models.extract_NAS_trees import *

import pandas as pd
import numpy as np 
import matplotlib.pyplot as plt
import pickle

from pathlib import Path
parent_dir = str(Path.cwd().parent)

## Tree NAS Run Diagnostics

## Tree NAS Model Selection

Retrieving models from NAS History Dict

In [ ]:
history_dict, current_episode = load_history(
        parent_dir+'/Results_Evaluation/WV_GED_NAS_Sampling_History.pkl'
    )

print(f"Loaded history up to episode {current_episode}, "
          f"{len(_episode_indices(history_dict))} episodes present.\n")

# --- diagnose reward distribution first (is it clustered at a ceiling?) ---
diagnose_reward_distribution(history_dict, reward_mode='second')

# pool from the highest-convergence episodes, wide enough to capture patterns
pool = extract_top_k(history_dict, k=10, min_convergence=0.0,
                                        reward_mode='sum', dedupe=False)

# collapse to distinct module patterns, best of each, keep top 10
distinct10 = extract_best_per_module_pattern(pool, k=10)
summarize_top_k(distinct10)
ensemble_top_k = get_top_k_configs(pool)

# save
top_k_configs = ensemble_top_k
# with open(parent_dir+'/Results_Evaluation/top10_ensemble_configs.pkl', 'wb') as f:
#     pickle.dump(top_k_configs, f)


count_module_patterns(pool)

Loaded history up to episode 1312, 1000 episodes present.


=== reward distribution (second subset, 10000 finite trees) ===
  min 0.000  p25 2.557  median 2.667  p75 2.741  max 2.989
  fraction within 5% of max (2.989): 3.6%  (spread out -> reward-ranking is informative)
    [ 0.000,  2.557): 25.0%
    [ 2.557,  2.667): 25.0%
    [ 2.667,  2.741): 25.0%
    [ 2.741,  2.989): 25.0%
rank    score   reward    conv    ep  nodes  modules
   1   5.7264   5.7264   0.520   934      3  Dnl_avgpool1d,Dnl_conv1d,Dnl_conv1d
   2   5.7059   5.7059   0.532   936      3  Dnl_avgpool1d,MTSMixer,MTSMixer
   3   5.6634   5.6634   0.376   863      3  Dnl_avgpool1d,Patching,Patching
   4   5.6601   5.6601   0.514   933      3  Dnl_avgpool1d,Dnl_avgpool1d,MTSMixer
   5   5.6579   5.6579   0.491   906      3  MTSMixer,Dnl_avgpool1d,MTSMixer
   6   5.6491   5.6491   0.412   581      3  Dnl_avgpool1d,MTSMixer,Patching
   7   5.6355   5.6355   0.511   426      3  Dnl_avgpool1d,TSMixer,Dnl_conv1d
   8   5.6334 

{'n_configs': 10, 'n_patterns': 9}

In [6]:
print(summarize_top_k(pool).groupby(['nodes']).score.agg(['mean', 'min', 'max','median', 'count']))

rank    score   reward    conv    ep  nodes  modules
   1   5.7264   5.7264   0.520   934      3  Dnl_avgpool1d,Dnl_conv1d,Dnl_conv1d
   2   5.7059   5.7059   0.532   936      3  Dnl_avgpool1d,MTSMixer,MTSMixer
   3   5.6634   5.6634   0.376   863      3  Dnl_avgpool1d,Patching,Patching
   4   5.6601   5.6601   0.514   933      3  Dnl_avgpool1d,Dnl_avgpool1d,MTSMixer
   5   5.6579   5.6579   0.491   906      3  MTSMixer,Dnl_avgpool1d,MTSMixer
   6   5.6491   5.6491   0.412   581      3  Dnl_avgpool1d,MTSMixer,Patching
   7   5.6355   5.6355   0.511   426      3  Dnl_avgpool1d,TSMixer,Dnl_conv1d
   8   5.6334   5.6334   0.498   509      3  Dnl_avgpool1d,Patching,MTSMixer
   9   5.6333   5.6333   0.488   904      3  MTSMixer,MTSMixer,MTSMixer
  10   5.6307   5.6307   0.534   954      3  Dnl_avgpool1d,Dnl_avgpool1d,MTSMixer
  11   5.6277   5.6277   0.455   885      3  Dnl_avgpool1d,MTSMixer,MTSMixer
  12   5.6243   5.6243   0.411   782      3  MTSMixer,MTSMixer,Dnl_conv1d
  13   5.6219   

In [15]:
pool = extract_top_k(history_dict, k=10, min_convergence=0.0,
                                        reward_mode='sum', dedupe=False)
# collapse to distinct module patterns, best of each, keep top 10
distinct10 = extract_best_per_module_pattern(pool, k=10)
summarize_top_k(pool)
summarize_top_k(distinct10)
count_module_patterns(pool)

rank    score   reward    conv    ep  nodes  modules
   1   5.7264   5.7264   0.520   934      3  Dnl_avgpool1d,Dnl_conv1d,Dnl_conv1d
   2   5.7059   5.7059   0.532   936      3  Dnl_avgpool1d,MTSMixer,MTSMixer
   3   5.6634   5.6634   0.376   863      3  Dnl_avgpool1d,Patching,Patching
   4   5.6601   5.6601   0.514   933      3  Dnl_avgpool1d,Dnl_avgpool1d,MTSMixer
   5   5.6579   5.6579   0.491   906      3  MTSMixer,Dnl_avgpool1d,MTSMixer
   6   5.6491   5.6491   0.412   581      3  Dnl_avgpool1d,MTSMixer,Patching
   7   5.6355   5.6355   0.511   426      3  Dnl_avgpool1d,TSMixer,Dnl_conv1d
   8   5.6334   5.6334   0.498   509      3  Dnl_avgpool1d,Patching,MTSMixer
   9   5.6333   5.6333   0.488   904      3  MTSMixer,MTSMixer,MTSMixer
  10   5.6307   5.6307   0.534   954      3  Dnl_avgpool1d,Dnl_avgpool1d,MTSMixer
rank    score   reward    conv    ep  nodes  modules
   1   5.7264   5.7264   0.520   934      3  Dnl_avgpool1d,Dnl_conv1d,Dnl_conv1d
   2   5.7059   5.7059   0.532   

{'n_configs': 10, 'n_patterns': 9}

## Running Selected Models from Tree NAS Benchmark

In [3]:

nas_benchmark_point = NAS_TopModels_Runner(model_configs=top_k_configs)


In [4]:
nas_benchmark_point.run_top_nas_forecasters()

Current Frame Name:  EU_Energy_Prices_Time_Series_
Current number of Input and Output Timesteps:  192 

Client dashboard link: http://127.0.0.1:8787/status


/home/RDC/khankiss/Dokumente/NAS_GP/.venv/lib/python3.11/site-packages/distributed/client.py:3357: UserWarning: Sending large graph of size 1.50 GiB.
This may cause some slowdown.
Consider scattering data ahead of time and using futures.
  warnings.warn(


Use GPU: cuda:0
Use GPU: cuda:0
Use GPU: cuda:0
Use GPU: cuda:0
Use GPU: cuda:0
Use GPU: cuda:0
Use GPU: cuda:0
Use GPU: cuda:0
Use GPU: cuda:0
Use GPU: cuda:0
Current Frame Name:  EU_Energy_Prices_Time_Series_
Current number of Input and Output Timesteps:  192 

Client dashboard link: http://127.0.0.1:8787/status


/home/RDC/khankiss/Dokumente/NAS_GP/.venv/lib/python3.11/site-packages/distributed/client.py:3357: UserWarning: Sending large graph of size 1.50 GiB.
This may cause some slowdown.
Consider scattering data ahead of time and using futures.
  warnings.warn(


Use GPU: cuda:0
Use GPU: cuda:0
Use GPU: cuda:0
Use GPU: cuda:0
Use GPU: cuda:0
Use GPU: cuda:0
Use GPU: cuda:0
Use GPU: cuda:0
Use GPU: cuda:0
Use GPU: cuda:0
Current Frame Name:  EU_Energy_Prices_Time_Series_
Current number of Input and Output Timesteps:  192 

Client dashboard link: http://127.0.0.1:8787/status


...

Use GPU: cuda:0
Use GPU: cuda:0
Use GPU: cuda:0
Use GPU: cuda:0
Use GPU: cuda:0
Use GPU: cuda:0
Use GPU: cuda:0
Use GPU: cuda:0
Use GPU: cuda:0
Use GPU: cuda:0
Current Frame Name:  EU_Energy_Prices_Time_Series_
Current number of Input and Output Timesteps:  192 

Client dashboard link: http://127.0.0.1:8787/status


/home/RDC/khankiss/Dokumente/NAS_GP/.venv/lib/python3.11/site-packages/distributed/client.py:3357: UserWarning: Sending large graph of size 1.50 GiB.
This may cause some slowdown.
Consider scattering data ahead of time and using futures.
  warnings.warn(


Use GPU: cuda:0
Use GPU: cuda:0
Use GPU: cuda:0
Use GPU: cuda:0
Use GPU: cuda:0
Use GPU: cuda:0
Use GPU: cuda:0
Use GPU: cuda:0
Use GPU: cuda:0
Use GPU: cuda:0
Current Frame Name:  EU_Energy_Prices_Time_Series_
Current number of Input and Output Timesteps:  192 

Client dashboard link: http://127.0.0.1:8787/status


/home/RDC/khankiss/Dokumente/NAS_GP/.venv/lib/python3.11/site-packages/distributed/client.py:3357: UserWarning: Sending large graph of size 1.50 GiB.
This may cause some slowdown.
Consider scattering data ahead of time and using futures.
  warnings.warn(


Use GPU: cuda:0
Use GPU: cuda:0
Use GPU: cuda:0
Use GPU: cuda:0
Use GPU: cuda:0
Use GPU: cuda:0
Use GPU: cuda:0
Use GPU: cuda:0
Use GPU: cuda:0
Use GPU: cuda:0


In [19]:
import Src.point_metrics_v2 as pm

trees = range(10)
seq_lens = [48, 96, 192]

# one tree, one run, to check paths and shapes before the full pass
d = pm.load_run(parent_dir, 0, 48, 1)
print(len(d), 'splits')
print(next(iter(d.values()))[0].shape)     # expect (n_sequences, 48, 19)

# all trees
df = pm.tree_table(trees, seq_lens, parent_dir)
df.to_pickle('point_metrics_long.pkl')
print(pm.summarise(df).round(4))

# ensemble, formed within each run
ens = pm.ensemble_table(trees, seq_lens, parent_dir)
print(pm.summarise(ens).round(4))

both = pd.concat([df, ens], ignore_index=True)
both.to_pickle('point_metrics_long.pkl')

17 splits
(308, 48, 19)
                RMSE     MSE     MAE   MedAE  RMSE_sd_run  MSE_sd_run  \
tree horizon                                                            
0    48       0.8804  0.8472  0.6074  0.4393       0.0130      0.0306   
     96       0.8912  0.8342  0.6282  0.4680       0.0079      0.0214   
     192      0.9590  1.0034  0.6906  0.5251       0.0199      0.0584   
1    48       0.8939  0.8726  0.6201  0.4503       0.0316      0.0628   
     96       0.9155  0.8769  0.6478  0.4827       0.0325      0.0683   
     192      0.9433  0.9781  0.6718  0.5073       0.0407      0.0838   
2    48       0.8732  0.8305  0.5987  0.4298       0.0081      0.0206   
     96       0.9212  0.8904  0.6543  0.4899       0.0278      0.0591   
     192      0.9539  0.9891  0.6817  0.5171       0.0335      0.0694   
3    48       0.8978  0.8771  0.6247  0.4565       0.0213      0.0448   
     96       0.9303  0.9083  0.6608  0.4959       0.0243      0.0476   
     192      0.9466  0.972

In [20]:
keep = [f'Train_Test{i}' for i in range(1, 16)]
ens = ens[ens.split.isin(keep)]
df  = df[df.split.isin(keep)]
order = sorted(ens.split.unique(), key=lambda s: int(s.replace('Train_Test', '')))
tab = ens.groupby(['horizon', 'split']).RMSE.mean().unstack()[order]
tab

split,Train_Test1,Train_Test2,Train_Test3,Train_Test4,Train_Test5,Train_Test6,Train_Test7,Train_Test8,Train_Test9,Train_Test10,Train_Test11,Train_Test12,Train_Test13,Train_Test14,Train_Test15
horizon,,,,,,,,,,,,,,,
48,0.918315,0.947009,1.035247,1.375683,0.610896,0.683857,1.021688,0.698735,0.724981,0.656342,1.112084,0.717560,1.305222,0.558419,0.870186
96,0.920558,1.003676,1.202208,0.822416,0.696848,1.093610,0.977493,0.715170,1.305109,0.745962,0.909749,0.648401,0.694145,0.749066,0.731727
192,1.102977,1.305923,0.770604,0.908329,0.692786,1.187497,1.388812,0.771412,0.830809,0.803193,0.948934,0.684730,0.613598,0.799392,0.569346


In [21]:
pm.summarise(pd.concat([df, ens], ignore_index=True)).round(4)

RMSE     MSE     MAE   MedAE  RMSE_sd_run  MSE_sd_run  \
tree     horizon                                                            
0        48       0.9122  0.9045  0.6278  0.4543       0.0144      0.0345   
         96       0.8972  0.8481  0.6315  0.4703       0.0090      0.0245   
         192      0.9590  1.0034  0.6906  0.5251       0.0199      0.0584   
1        48       0.9275  0.9329  0.6424  0.4672       0.0314      0.0652   
         96       0.9172  0.8825  0.6469  0.4804       0.0322      0.0681   
         192      0.9433  0.9781  0.6718  0.5073       0.0407      0.0838   
2        48       0.9060  0.8878  0.6198  0.4452       0.0090      0.0231   
         96       0.9244  0.8989  0.6549  0.4899       0.0297      0.0631   
         192      0.9539  0.9891  0.6817  0.5171       0.0335      0.0694   
3        48       0.9314  0.9372  0.6474  0.4741       0.0226      0.0488   
         96       0.9311  0.9123  0.6591  0.4928       0.0238      0.0468   
         192      0.9466  0.9729  0.6728  0.5092       0.0105      0.0348   
4        48       0.9133  0.9005  0.6276  0.4485       0.0113      0.0248   
         96       0.9114  0.8715  0.6395  0.4714       0.0161      0.0302   
         192      0.9219  0.9328  0.6432  0.4768       0.0238      0.0748   
5        48       0.9087  0.8893  0.6236  0.4495       0.0062      0.0123   
         96       0.9193  0.8884  0.6469  0.4793       0.0116      0.0239   
         192      0.9472  0.9700  0.6695  0.5005       0.0104      0.0250   
6        48       0.9007  0.8714  0.6153  0.4452       0.0027      0.0055   
         96       0.8910  0.8331  0.6272  0.4695       0.0085      0.0173   
         192      0.9606  0.9917  0.6939  0.5336       0.0170      0.0408   
7        48       0.9052  0.8840  0.6248  0.4529       0.0131      0.0276   
         96       0.9152  0.8802  0.6434  0.4776       0.0215      0.0438   
         192      0.9545  0.9868  0.6751  0.5071       0.0223      0.0590   
8        48       0.9075  0.8870  0.6237  0.4446       0.0024      0.0061   
         96       0.8969  0.8391  0.6243  0.4558       0.0042      0.0082   
         192      0.8916  0.8656  0.6189  0.4504       0.0040      0.0126   
9        48       0.9292  0.9366  0.6440  0.4709       0.0265      0.0589   
         96       0.9350  0.9165  0.6608  0.4920       0.0126      0.0227   
         192      0.9627  1.0437  0.6897  0.5233       0.0193      0.0897   
ensemble 48       0.8824  0.8372  0.5990  0.4280       0.0020      0.0039   
         96       0.8811  0.8142  0.6143  0.4517       0.0053      0.0103   
         192      0.8919  0.8540  0.6210  0.4582       0.0064      0.0143   

                  MAE_sd_run  MedAE_sd_run  RMSE_sd_split  MSE_sd_split  \
tree     horizon                                                          
0        48           0.0121        0.0119         0.2759        0.5649   
         96           0.0105        0.0117         0.2118        0.4175   
         192          0.0166        0.0141         0.2910        0.6491   
1        48           0.0338        0.0402         0.2753        0.5664   
         96           0.0352        0.0382         0.2039        0.4019   
         192          0.0426        0.0477         0.2928        0.6459   
2        48           0.0055        0.0054         0.2669        0.5368   
         96           0.0313        0.0337         0.2083        0.4161   
         192          0.0352        0.0434         0.2780        0.5971   
3        48           0.0216        0.0239         0.2706        0.5495   
         96           0.0197        0.0141         0.2088        0.4147   
         192          0.0122        0.0164         0.2799        0.6094   
4        48           0.0126        0.0121         0.2659        0.5335   
         96           0.0144        0.0125         0.2054        0.4045   
         192          0.0192        0.0176         0.2828        0.6147   
5        48           0.0081        0.0086       

### Hold out Ensembles

In [23]:
for held_out in trees:
    members = [t for t in trees if t != held_out]
    e = pm.ensemble_table(members, seq_lens, parent_dir)
    e = e[e.split.isin(keep)]
    print(held_out, pm.summarise(e).RMSE.values)
    

0 [0.88412807 0.8839588  0.89472159]
1 [0.88088784 0.88010387 0.89164245]
2 [0.88288172 0.88003204 0.89085623]
3 [0.8811499  0.87926058 0.89073237]
4 [0.88252838 0.88088891 0.89405955]
5 [0.88314093 0.8806372  0.89122183]
6 [0.88480134 0.88647161 0.89306797]
7 [0.88327496 0.88152431 0.89248671]
8 [0.88375394 0.88301115 0.89662243]
9 [0.88161765 0.87902081 0.89025039]


### Full ensemble

In [24]:
full_ens = pm.ensemble_table(trees, seq_lens, parent_dir)
full_ens = full_ens[full_ens.split.isin(keep)]
print('ens', pm.summarise(full_ens).RMSE.values)

ens [0.88241487 0.88107595 0.89188946]


### Results of individual trees

In [25]:
df[(df.horizon == 192)].groupby(['tree', 'split']).RMSE.mean().unstack()[order]

split,Train_Test1,Train_Test2,Train_Test3,Train_Test4,Train_Test5,Train_Test6,Train_Test7,Train_Test8,Train_Test9,Train_Test10,Train_Test11,Train_Test12,Train_Test13,Train_Test14,Train_Test15
tree,,,,,,,,,,,,,,,
0,1.240415,1.453774,0.793330,0.957711,0.718584,1.177459,1.607654,0.799383,0.848845,0.872893,1.011422,0.791815,0.668062,0.853676,0.590063
1,1.161370,1.590995,0.824833,0.959217,0.723736,1.261223,1.422528,0.763198,0.842911,0.816942,0.988802,0.738773,0.644026,0.836856,0.573442
2,1.192952,1.509184,0.864026,0.953209,0.730367,1.296494,1.412090,0.814234,0.868123,0.814696,1.014911,0.760418,0.640565,0.851390,0.586473
3,1.140423,1.544819,0.831753,0.949401,0.720205,1.259230,1.426596,0.780107,0.858380,0.820989,1.002403,0.727702,0.653791,0.877528,0.605727
4,1.134538,1.516154,0.779412,0.921337,0.705029,1.191056,1.439767,0.793372,0.851730,0.795682,0.989641,0.697311,0.616088,0.817035,0.580965
5,1.262315,1.438366,0.908672,0.941464,0.717825,1.249055,1.397870,0.808921,0.830926,0.831866,1.080181,0.721285,0.627916,0.819765,0.572164
6,1.223715,1.450600,0.801849,0.966092,0.770144,1.263405,1.430203,0.844851,0.859248,0.851018,1.066550,0.771225,0.646225,0.854013,0.609787
7,1.184837,1.437146,0.798530,0.957500,0.712480,1.291282,1.465669,0.806086,0.849488,0.922001,1.033440,0.769147,0.646363,0.835979,0.607318
8,1.085697,1.497760,0.787009,0.921952,0.699949,1.089785,1.420465,0.785672,0.801490,0.773746,0.888259,0.700685,0.603748,0.760204,0.558189


## Run Chain NAS

Chain models are run directly by running the script "NAS_Chain_Models.py" in Benchmark_Models folder

In [ ]:
history_dict_l, current_episode = load_history(
        '/home/RDC/khankiss/H/NAS_run_dicts/WV_GED_NAS_Sampling_History_8_999_linear.pkl'
    )

print(f"Loaded history up to episode {current_episode}, "
          f"{len(_episode_indices(history_dict_l))} episodes present.\n")

# --- diagnose reward distribution first (is it clustered at a ceiling?) ---
diagnose_reward_distribution(history_dict_l, reward_mode='second')

# pool from the highest-convergence episodes, wide enough to capture patterns
pool_linear = extract_top_k(history_dict_l, k=10, min_convergence=0.0,
                                        reward_mode='sum', dedupe=False, exclude_reward=6.0)
# collapse to distinct module patterns, best of each, keep top 10
distinct10_l = extract_best_per_module_pattern(pool_linear, k=10)
summarize_top_k(distinct10_l)
ensemble_top_k_l = get_top_k_configs(pool_linear)
# save
top_k_configs_l = ensemble_top_k_l
# with open(parent_dir+'/Results_Evaluation/top10_ensemble_configs_linear.pkl', 'wb') as f:
#     pickle.dump(top_k_configs_l, f)


count_module_patterns(pool_linear)

In [ ]:
summarize_top_k(pool_linear).groupby(['nodes']).score.agg(['mean', 'min', 'max','median', 'count'])

In [3]:
pool_linear = extract_top_k(history_dict_l, k=10, min_convergence=0.0,
                                        reward_mode='sum', dedupe=False, exclude_reward=6.0)
# collapse to distinct module patterns, best of each, keep top 10
distinct10_l = extract_best_per_module_pattern(pool_linear, k=10)
summarize_top_k(pool_linear)
summarize_top_k(distinct10_l)
count_module_patterns(pool_linear)

rank    score   reward    conv    ep  nodes  modules
   1   5.7534   5.7534   0.450   195      3  Dnl_conv1d,TSMixer,TSMixer
   2   5.6673   5.6673   0.644   627      2  Dnl_avgpool1d,MTSMixer
   3   5.6551   5.6551   0.659   651      3  MTSMixer,Dnl_conv1d,MTSMixer
   4   5.6532   5.6532   0.627   924      2  MTSMixer,Dnl_avgpool1d
   5   5.6501   5.6501   0.588   960      2  MTSMixer,Dnl_avgpool1d
   6   5.6477   5.6477   0.543   772      2  MTSMixer,Dnl_conv1d
   7   5.6444   5.6444   0.522   231      2  Dnl_conv1d,Dnl_avgpool1d
   8   5.6443   5.6443   0.661   666      3  TSMixer,MTSMixer,TSMixer
   9   5.6430   5.6430   0.480   301      2  MTSMixer,Dnl_avgpool1d
  10   5.6396   5.6396   0.352   107      2  Dnl_conv1d,Dnl_conv1d
rank    score   reward    conv    ep  nodes  modules
   1   5.7534   5.7534   0.450   195      3  Dnl_conv1d,TSMixer,TSMixer
   2   5.6673   5.6673   0.644   627      2  Dnl_avgpool1d,MTSMixer
   3   5.6551   5.6551   0.659   651      3  MTSMixer,Dnl_conv1d

{'n_configs': 10, 'n_patterns': 8}

It can also be run through ipynb

In [ ]:
nas_benchmark_point_linear = NAS_Chain_Runner(model_configs=top_k_configs_l)
nas_benchmark_point_linear.run_top_nas_forecasters()

Save Point Metric Results

In [ ]:
import Src.point_metrics_v2 as pm

chains = range(10)
seq_lens = [48, 96, 192]

# one tree, one run, to check paths and shapes before the full pass
d = pm.load_run(parent_dir, 0, 48, 1)
print(len(d), 'splits')
print(next(iter(d.values()))[0].shape)     # expect (n_sequences, 48, 19)

# all trees
df_l = pm.tree_table(chains, seq_lens, parent_dir)
print(pm.summarise(df_l).round(4))

# ensemble, formed within each run
ens_l = pm.ensemble_table(chains, seq_lens, parent_dir)
print(pm.summarise(ens_l).round(4))

both_l = pd.concat([df_l, ens_l], ignore_index=True)
both_l.to_pickle('point_metrics_chain_long.pkl')

In [63]:
keep = [f'Train_Test{i}' for i in range(1, 16)]
both_l = both_l[both_l.split.isin(keep)]
order_l = sorted(both_l.split.unique(), key=lambda s: int(s.replace('Train_Test', '')))
both_l.loc[(both_l.horizon == 192) & (both_l.tree != 2) & (both_l.tree != 7)].groupby(['tree', 'split']).RMSE.mean().unstack()[order_l]

split,Train_Test1,Train_Test2,Train_Test3,Train_Test4,Train_Test5,Train_Test6,Train_Test7,Train_Test8,Train_Test9,Train_Test10,Train_Test11,Train_Test12,Train_Test13,Train_Test14,Train_Test15
tree,,,,,,,,,,,,,,,
0,1.844588e+07,1.000000e+00,1.000000,1.000000e+00,1.000000e+00,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000
1,1.175119e+00,1.499501e+00,0.809168,9.218753e-01,7.153626e-01,1.471919,1.561614,0.771320,0.835597,0.846243,1.028378,0.706288,0.616348,0.812510,0.571688
3,1.084350e+00,1.388700e+00,0.965638,1.808727e+00,1.843574e+00,2.049582,3.003823,1.897905,1.898891,2.849606,5.352484,1.717827,1.823106,2.150119,0.929865
4,1.196301e+00,1.469552e+00,1.224016,1.464123e+00,1.512645e+00,2.380293,1.895912,1.546272,1.730409,2.991679,2.950262,4.347526,2.633113,4.162735,2.476230
5,1.225252e+00,1.665529e+00,0.825722,9.615612e-01,7.346687e-01,1.260213,1.514936,0.836405,0.842450,0.874144,1.129041,0.756379,0.659573,0.821384,0.591138
6,1.095742e+00,1.468408e+00,0.807748,9.716306e-01,7.531257e-01,1.053065,1.549248,0.829830,0.740084,358.952369,0.921461,0.919470,0.817732,0.849340,0.785565
8,1.220522e+00,1.409008e+00,0.813174,9.307703e-01,7.202226e-01,1.262078,1.516575,0.860617,0.893681,0.910455,1.160894,0.789781,0.636004,0.887391,0.619488
9,4.146711e+06,6.471668e+07,71800.737006,6.720423e+06,5.407386e+08,1.076866,109.902475,337886.090328,158887.567372,61099.643538,1.000000,1.000000,1.000000,1.000000,1.000000
ensemble,1.831833e+06,1.000000e+00,1.000000,1.000000e+00,1.000000e+00,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000


### Filtering out explosive Chain runs and corresponding models in ensemble

In [ ]:
import point_metrics_v2 as pm

chain = pm.tree_table(range(10), [48, 96, 192], '/home/RDC/khankiss/H', prefix='chain')
clean, summary = pm.filter_runs(chain)
print(summary)

ens_chain = pm.ensemble_filtered(clean, [48, 96, 192], '/home/RDC/khankiss/H', pm.load_run, pm.split_metrics, prefix='chain')
print(pm.summarise(pd.concat([clean,ens_chain], ignore_index=True)).round(4))


              discarded  total  surviving
horizon tree                             
48      0            10     10          0
        1             0     10         10
        2             9     10          1
        3             0     10         10
        4             0     10         10
        5             0     10         10
        6             0     10         10
        7            10     10          0
        8             0     10         10
        9             2     10          8
96      0            10     10          0
        1             0     10         10
        2            10     10          0
        3             0     10         10
        4             0     10         10
        5             0     10         10
        6             0     10         10
        7            10     10          0
        8             0     10         10
        9            10     10          0
192     0            10     10          0
        1             0     10    

Save clean results

In [4]:
chain_long_results = pd.concat([clean,ens_chain], ignore_index=True)
chain_long_results.to_pickle('point_metrics_chain_long.pkl')

Summarise results

In [ ]:
pm.summarise(pd.concat([clean, ens_chain], ignore_index=True))

RMSE        MSE       MAE     MedAE  RMSE_sd_run  \
tree     horizon                                                         
1        48       0.863256   0.808087  0.597079  0.432369     0.002895   
         96       0.887844   0.823255  0.626661  0.466718     0.002329   
         192      0.956195   1.016209  0.689412  0.524974     0.020666   
2        48       5.416856  62.624360  1.807796  1.029947          NaN   
3        48       0.930209   0.935653  0.624783  0.415686     0.045315   
         96       1.070124   1.371082  0.733204  0.512577     0.097303   
         192      2.050947  12.484294  1.365419  0.889005     0.962499   
4        48       0.921198   0.910135  0.619735  0.411948     0.013957   
         96       1.015262   1.092507  0.699781  0.493480     0.041929   
         192      2.265405   9.187782  1.422488  0.881626     0.894238   
5        48       0.861493   0.806528  0.593689  0.429038     0.003325   
         96       0.896443   0.838640  0.633541  0.473904     0.010313   
         192      0.979893   1.056972  0.710614  0.546173     0.017442   
6        48       0.909539   0.896468  0.618937  0.416000     0.002862   
         96       0.930493   0.898124  0.642645  0.453052     0.016120   
         192      0.912980   0.906075  0.627996  0.437560     0.007344   
8        48       0.865969   0.814383  0.599113  0.436081     0.002122   
         96       0.895195   0.838726  0.632906  0.474013     0.003712   
         192      0.975377   1.024300  0.694414  0.521376     0.025255   
9        48       1.404712  27.134094  0.730820  0.461818     1.444771   
ensemble 48       0.918574   1.278484  0.578829  0.390109     0.184300   
         96       0.871579   0.788214  0.598661  0.428367     0.007874   
         192      1.073176   1.372721  0.732233  0.508440     0.142358   

                  MSE_sd_run  MAE_sd_run  MedAE_sd_run  RMSE_sd_split  \
tree     horizon                                                        
1        48         0.006488    0.002157      0.002145       0.258360   
         96         0.004117    0.002040      0.001856       0.192651   
         192        0.061067    0.020452      0.020490       0.323912   
2        48              NaN         NaN           NaN       5.946609   
3        48         0.115905    0.028969      0.018553       0.238658   
         96         0.422651    0.059096      0.034892       0.229934   
         192       20.786369    0.631327      0.385164       1.088069   
4        48         0.027044    0.010244      0.006990       0.252011   
         96         0.103347    0.029143      0.021100       0.179009   
         192        7.535537    0.514753      0.281484       1.004771   
5        48         0.007603    0.001992      0.001728       0.261332   
         96         0.018899    0.008870      0.008226       0.191536   
         192        0.042856    0.017365      0.019342       0.314088   
6        48         0.006598    0.002590      0.002223       0.271015   
         96         0.032937    0.010358      0.008013       0.180785   
         192        0.013118    0.005959      0.005940       0.278190   
8        48         0.005149    0.001424      0.001726       0.261657   
         96         0.006707    0.002753      0.002050       0.198914   
         192        0.053304    0.021074      0.018872       0.275048   
9        48        74.305484    0.304742      0.019979       1.697438   
ensemble 48         1.351166    0.040242      0.013199       0.259019   
         96         0.013426    0.005799      0.004130       0.171493   
         192        0.559547    0.097357      0.067959       0.243126   

                  MSE_sd_split  MAE_sd_split  MedAE_sd_split  
tree     horizon                                              
1        48           0.494849      0.141985        0.098368  
         96           0.364622      0.116645        0.098204  
         192          0.716138      0.232441        0.194360  
2        48         157.075956

Per split RMSE of individual Chain NAS and ensemble models

In [9]:
keep = [f'Train_Test{i}' for i in range(1, 16)]
chain_long_results = chain_long_results[chain_long_results.split.isin(keep)]
order_l = sorted(chain_long_results.split.unique(), key=lambda s: int(s.replace('Train_Test', '')))
chain_long_results[chain_long_results.horizon == 192].groupby(['tree', 'split']).RMSE.mean().unstack()[order_l].round(4)

split,Train_Test1,Train_Test2,Train_Test3,Train_Test4,Train_Test5,Train_Test6,Train_Test7,Train_Test8,Train_Test9,Train_Test10,Train_Test11,Train_Test12,Train_Test13,Train_Test14,Train_Test15
tree,,,,,,,,,,,,,,,
1,1.1751,1.4995,0.8092,0.9219,0.7154,1.4719,1.5616,0.7713,0.8356,0.8462,1.0284,0.7063,0.6163,0.8125,0.5717
3,1.0844,1.3887,0.9656,1.8087,1.8436,2.0496,3.0038,1.8979,1.8989,2.8496,5.3525,1.7178,1.8231,2.1501,0.9299
4,1.1963,1.4696,1.2240,1.4641,1.5126,2.3803,1.8959,1.5463,1.7304,2.9917,2.9503,4.3475,2.6331,4.1627,2.4762
5,1.2253,1.6655,0.8257,0.9616,0.7347,1.2602,1.5149,0.8364,0.8425,0.8741,1.1290,0.7564,0.6596,0.8214,0.5911
6,1.0971,1.4649,0.8074,0.9723,0.7521,1.0539,1.5503,0.8273,0.7404,0.9237,0.7947,0.7311,0.6288,0.6877,0.6630
8,1.2205,1.4090,0.8132,0.9308,0.7202,1.2621,1.5166,0.8606,0.8937,0.9105,1.1609,0.7898,0.6360,0.8874,0.6195
ensemble,1.0762,1.2635,0.7985,0.9402,0.8016,1.1939,1.5174,0.9156,0.8252,1.1936,1.5277,1.1210,0.8956,1.1930,0.8347


This is to verify that there are no explosive runs in Tree models

In [ ]:

trees = pm.tree_table(range(10), [48, 96, 192], '/home/RDC/khankiss/H', prefix='tree')
clean, summary = pm.filter_runs(trees)
print(summary)

ens_trees = pm.ensemble_filtered(clean, [48, 96, 192], '/home/RDC/khankiss/H', pm.load_run, pm.split_metrics, prefix='tree')
print(pm.summarise(pd.concat([clean,ens_trees], ignore_index=True)).round(4))

              discarded  total  surviving
horizon tree                             
48      0             0     10         10
        1             0     10         10
        2             0     10         10
        3             0     10         10
        4             0     10         10
        5             0     10         10
        6             0     10         10
        7             0     10         10
        8             0     10         10
        9             0     10         10
96      0             0     10         10
        1             0     10         10
        2             0     10         10
        3             0     10         10
        4             0     10         10
        5             0     10         10
        6             0     10         10
        7             0     10         10
        8             0     10         10
        9             0     10         10
192     0             0     10         10
        1             0     10    

## Full NAS Tree ensemble (10 architectures) vs. Top 6 NAS Tree architectures

change the file directories accordingly to reproduce results

In [3]:
top6 = [0, 1, 2, 3, 4, 5]
trees_top6 = pm.tree_table(top6, [48, 96, 192], '/home/RDC/khankiss/H', prefix='tree')
clean, summary = pm.filter_runs(trees_top6)
print(summary)

ens_trees_top6 = pm.ensemble_filtered(clean, [48, 96, 192], '/home/RDC/khankiss/H', pm.load_run, pm.split_metrics, prefix='tree')
print(pm.summarise(pd.concat([clean,ens_trees_top6], ignore_index=True)).round(4))

              discarded  total  surviving
horizon tree                             
48      0             0     10         10
        1             0     10         10
        2             0     10         10
        3             0     10         10
        4             0     10         10
        5             0     10         10
96      0             0     10         10
        1             0     10         10
        2             0     10         10
        3             0     10         10
        4             0     10         10
        5             0     10         10
192     0             0     10         10
        1             0     10         10
        2             0     10         10
        3             0     10         10
        4             0     10         10
        5             0     10         10
                    RMSE     MSE     MAE   MedAE  RMSE_sd_run  MSE_sd_run  \
tree     horizon                                                            
0     

## NAS Benchmarking (Pretrained and Multivariate baselines, AutoGluon NAS)

This section aggregates all results together under one long dataframe for future analysis purposes. Files are stored in "Src/stored_metric_results".

Before aggregating the results, please run Baseline Benchmarks give in "Run_Baseline_Models.ipynb"

In [ ]:
import pandas as pd
import point_metrics_v2 as pm

# ag = pm.autogluon_table(parent_dir, folder_name='Results_Evaluation', seq_lens=[48, 96, 192], n_runs=10)
# ag.to_pickle('point_metrics_ag_long.pkl')
# pretrained = pm.pretrained_table(seq_lens=[48, 96, 192], parent_dir="/home/RDC/khankiss/H", folder_name="Pretrained_Benchmark")
# pretrained.to_pickle('point_metrics_pretrained_long.pkl')
# multivariate = pm.multivariate_table(seq_lens=[48, 96, 192], parent_dir="/home/RDC/khankiss/H", folder_name="Benchmark_Multivariate")
# multivariate.to_pickle('point_metrics_multivariate_long.pkl')
with open(parent_dir+'/Src/stored_metric_results/point_metrics_chain_long.pkl', 'rb') as f:
    chain_long_results = pickle.load(f)
with open(parent_dir+'/Src/stored_metric_results/point_metrics_long.pkl', 'rb') as f:
    tree_long_results = pickle.load(f)
with open(parent_dir+'/Src/stored_metric_results/point_metrics_ag_long.pkl', 'rb') as f:
    ag = pickle.load(f)
with open(parent_dir+'/Src/stored_metric_results/point_metrics_pretrained_long.pkl', 'rb') as f:
    pretrained = pickle.load(f)
with open(parent_dir+'/Src/stored_metric_results/point_metrics_multivariate_long.pkl', 'rb') as f:
    multivariate = pickle.load(f)

ag['arm'] = 'autogluon'
pretrained['arm'] = 'pretrained'
multivariate['arm'] = 'multivariate'
chain_long_results['arm'] = 'chain'
tree_long_results['arm'] = 'tree'
pm.summarise(pd.concat([chain_long_results, tree_long_results, ag, pretrained, multivariate]), by=('arm', 'tree', 'horizon')).round(4)

RMSE     MSE     MAE   MedAE  RMSE_sd_run  \
arm       tree      horizon                                                
autogluon AutoGluon 48       1.1428  1.4668  0.7445  0.4740       0.0000   
                    96       1.1506  1.4126  0.7595  0.4950       0.0000   
                    192      1.0184  1.1562  0.6879  0.4575       0.0000   
chain     1         48       0.8633  0.8081  0.5971  0.4324       0.0029   
                    96       0.8878  0.8233  0.6267  0.4667       0.0023   
...                             ...     ...     ...     ...          ...   
tree      9         96       0.9334  0.9114  0.6624  0.4952       0.0153   
                    192      0.9627  1.0437  0.6897  0.5233       0.0193   
          ensemble  48       0.8514  0.7845  0.5793  0.4136       0.0020   
                    96       0.8797  0.8093  0.6152  0.4535       0.0047   
                    192      0.8919  0.8540  0.6210  0.4582       0.0064   

                             MSE_sd_run  MAE_sd_run  MedAE_sd_run  \
arm       tree      horizon                                         
autogluon AutoGluon 48           0.0000      0.0000        0.0000   
                    96           0.0000      0.0000        0.0000   
                    192          0.0000      0.0000        0.0000   
chain     1         48           0.0065      0.0022        0.0021   
                    96           0.0041      0.0020        0.0019   
...                                 ...         ...           ...   
tree      9         96           0.0274      0.0172        0.0197   
                    192          0.0897      0.0128        0.0168   
          ensemble  48           0.0036      0.0025        0.0036   
                    96           0.0091      0.0048        0.0053   
                    192          0.0143      0.0048        0.0045   

                             RMSE_sd_split  MSE_sd_split  MAE_sd_split  \
arm       tree      horizon                                              
autogluon AutoGluon 48              0.4150        1.1632        0.2323   
                    96              0.3081        0.7742        0.1919   
                    192             0.3591        0.9359        0.2065   
chain     1         48              0.2584        0.4948        0.1420   
                    96              0.1927        0.3646        0.1166   
...                                    ...           ...           ...   
tree      9         96              0.2013        0.3987        0.1245   
                    192             0.3338        0.8209        0.2206   
          ensemble  48              0.2518        0.4758        0.1317   
                    96              0.1939        0.3703        0.1145   
                    192             0.2489        0.4918        0.1471   

                             MedAE_sd_split  
arm       tree      horizon                  
autogluon AutoGluon 48               0.1398  
                    96               0.1484  
                    192              0.1208  
chain     1         48               0.0984  
                    96               0.0982  
...                                     ...  
tree      9         96               0.1011  
                    192              0.1571  
          ensemble  48               0.0915  
                    96               0.0943  
                    192              0.1072  

[86 rows x 12 columns]

In [ ]:
all_results = pd.concat([chain_long_results, tree_long_results, ag, pretrained, multivariate], ignore_index=True)
all_results.to_pickle('stored_metric_results/point_metrics_all_benchmarks_long.pkl')
all_results.split.unique()

In [ ]:
with open('stored_metric_results/point_metrics_all_benchmarks_long.pkl', 'rb') as f:
    all_results = pickle.load(f)
keep = [f'Train_Test{i}' for i in range(1, 16)]
full = all_results[all_results.split.isin(keep)]
print(full.split.unique())
full = pm.summarise(full, by=('arm','tree', 'horizon')).round(4)

main = full.loc[full.index.get_level_values('tree').isin(
    ['ensemble', 'AutoGluon', 'TimeMoe', 'TimesFM', 'Sundial',
     'GPT4TS', 'Basisformer', 'TSMixer', 'MTSMixer', 'Dlinear', 'PatchTST']
)]
main[['RMSE', 'RMSE_sd_run', 'RMSE_sd_split']].round(4)

['Train_Test1' 'Train_Test2' 'Train_Test3' 'Train_Test4' 'Train_Test5'
 'Train_Test6' 'Train_Test7' 'Train_Test8' 'Train_Test9' 'Train_Test10'
 'Train_Test11' 'Train_Test12' 'Train_Test13' 'Train_Test14'
 'Train_Test15']


RMSE  RMSE_sd_run  RMSE_sd_split
arm          tree        horizon                                    
autogluon    AutoGluon   48       1.1588       0.0000         0.4258
                         96       1.1711       0.0000         0.3090
                         192      1.0184       0.0000         0.3591
chain        ensemble    48       0.9005       0.1079         0.2531
                         96       0.8703       0.0089         0.1790
                         192      1.0732       0.1424         0.2431
multivariate Basisformer 48       0.9579       0.0029         0.2743
                         96       0.9400       0.0045         0.1818
                         192      0.9174       0.0057         0.2644
             Dlinear     48       1.0159       0.0063         0.3227
                         96       0.9561       0.0023         0.2386
                         192      0.9417       0.0005         0.2817
             MTSMixer    48       1.1254       0.0030         0.3478
                         96       1.0664       0.0027         0.2605
                         192      1.1031       0.0074         0.3240
             PatchTST    48       0.9377       0.0005         0.2575
                         96       0.9215       0.0007         0.1865
                         192      0.8656       0.0007         0.2444
             TSMixer     48       0.9231       0.0028         0.2648
                         96       0.9149       0.0015         0.2155
                         192      0.9271       0.0010         0.2802
pretrained   GPT4TS      48       1.2072       0.0056         0.3570
                         96       1.1594       0.0060         0.2413
                         192      1.1374       0.0018         0.2646
             Sundial     48       0.9927       0.0004         0.2743
                         96       0.9527       0.0003         0.2182
                         192      0.8690       0.0002         0.2421
             TimeMoe     48       0.9402          NaN         0.2833
                         96       0.8712          NaN         0.2016
                         192      0.8563          NaN         0.2136
             TimesFM     48       1.0114          NaN         0.2719
                         96       0.9419          NaN         0.2009
                         192      0.8840          NaN         0.2330
tree         ensemble    48       0.8824       0.0020         0.2504
                         96       0.8811       0.0053         0.2012
                         192      0.8919       0.0064         0.2489

In [ ]:
print(main[['RMSE', 'MSE', 'MAE', 'MedAE']].round(4))

                                    RMSE     MSE     MAE   MedAE
arm          tree        horizon                                
autogluon    AutoGluon   48       1.1588  1.5113  0.7507  0.4747
                         96       1.1711  1.4602  0.7709  0.5017
                         192      1.0184  1.1562  0.6879  0.4575
chain        ensemble    48       0.9005  0.9160  0.5869  0.4009
                         96       0.8703  0.7880  0.5952  0.4247
                         192      1.0732  1.3727  0.7322  0.5084
multivariate Basisformer 48       0.9579  0.9879  0.6463  0.4333
                         96       0.9400  0.9147  0.6527  0.4672
                         192      0.9174  0.9071  0.6319  0.4463
             Dlinear     48       1.0159  1.1293  0.7039  0.5132
                         96       0.9561  0.9673  0.6787  0.5102
                         192      0.9417  0.9609  0.6657  0.4985
             MTSMixer    48       1.1254  1.3796  0.7764  0.5706
                         

In [ ]:
order_l = sorted(keep, key=lambda s: int(s.replace('Train_Test', '')))
all_results[
          all_results.split.isin(keep) 
          & all_results.horizon.isin([48, 96, 192]) 
          & all_results.tree.isin(['ensemble', 'AutoGluon', 'TimeMoe', 'TimesFM', 'Sundial', 'GPT4TS', 'Basisformer', 'TSMixer', 'MTSMixer', 'Dlinear', 'PatchTST'])
          ].groupby([ 'arm', 'tree', 'horizon','split']).RMSE.mean().unstack()[order_l].round(4)

split                             Train_Test1  Train_Test2  Train_Test3  \
arm          tree        horizon                                          
autogluon    AutoGluon   48               NaN       1.6216       2.2085   
                         96               NaN       1.8291       1.4449   
                         192              NaN       1.3256       1.1522   
chain        ensemble    48            0.8806       0.9914       1.0296   
                         96            0.8313       0.9909       1.1735   
                         192           1.0762       1.2635       0.7985   
multivariate Basisformer 48            0.9747       1.0571       1.0819   
                         96            0.9224       1.0793       1.2406   
                         192           1.2035       1.4530       0.8414   
             Dlinear     48            1.2191       1.2563       1.2925   
                         96            1.1407       1.2121       1.3664   
                         192           1.3985       1.4364       0.8566   
             MTSMixer    48            1.1891       1.2968       1.3804   
                         96            1.2207       1.3600       1.5549   
                         192           1.6459       1.6986       1.0579   
             PatchTST    48            0.9495       1.0979       1.1550   
                         96            0.8945       1.0519       1.2416   
                         192           1.0848       1.2848       0.7922   
             TSMixer     48            1.0733       1.0715       1.1382   
                         96            1.0449       1.0995       1.2607   
                         192           1.3121       1.4281       0.8196   
pretrained   GPT4TS      48            1.2135       1.4573       1.4821   
                         96            1.1882       1.4335       1.6583   
                         192           1.5500       1.6947       1.1013   
             Sundial     48            1.0208       1.1754       1.2300   
                         96            0.9011       1.0905       1.2923   
                         192           1.0896       1.2395       0.7946   
             TimeMoe     48            0.9471       1.0871       1.1906   
                         96            0.8651       1.0676       1.3192   
                         192           1.2035       1.0954       0.7948   
             TimesFM     48            1.0596       1.2702       1.2582   
                         96            0.9350       1.1195       1.3384   
                         192           1.0828       1.1793       0.8134   
tree         ensemble    48            0.9183       0.9470       1.0352   
                         96            0.9206       1.0037       1.2022   
                         192           1.1030       1.3059       0.7706   

split                             Train_Test4  Train_Test5  Train_Test6  \
arm          tree        horizon                                          
autogluon    AutoGluon   48            0.9811       1.1206       1.2005   
                         96            1.0718       1.5162       1.2616   
                         192           1.0504       1.2719       1.9847   
chain        ensemble    48            1.3377       0.6024       0.6860   
                         96            0.7911       0.6882       1.0856   
                         192           0.9402       0.8016       1.1939   
multivariate Basisformer 48            1.5020       0.6276       0.7276   
                         96            0.8722       0.7500       1.1806   
                         192           1.0004       0.7873       1.0441   
             Dlinear     48            1.6900       0.7435       0.7866   
                         96            0.9514       0.7624       1.1970   
                         192           0.9924       0.7643       1.1373   
             MTSMixer    48            1.9791       0.8583       0.8805   
                        

## Runtime Comparison

This section gathers computational runtime (Training + Inference runtime) required by Benchmarked models for a fair comparison under TOPSIS.

### Autogluon Runtime

In [33]:
import numpy as np
all_runtimes = []
for run in range(1, 11):
    with open(str(parent_dir) + '/Results_Evaluation/AutoGluonTS_NAS_Res_RMSE_' + str(run) + '.pkl', 'rb') as f:
        r = pickle.load(f)
    runtimes = []
    for seq_len in [48, 96, 192]: 
        runtime = r['NASTrain_Energy_Prices_Data'][seq_len]['Training_Testing_Time']
        runtimes.append(runtime)
    all_runtimes.append(runtimes)
    print(f"Run {run}: 48: {runtimes[0]}, 96: {runtimes[1]}, 192: {runtimes[2]}")
print(f"Average runtime across all runs: 48: {np.mean([r[0] for r in all_runtimes]).round(2)}, 96: {np.mean([r[1] for r in all_runtimes]).round(2)}, 192: {np.mean([r[2] for r in all_runtimes]).round(2)}")

Run 1: 48: 52.00511854422557, 96: 70.2837293024187, 192: 85.92308758605611
Run 2: 48: 51.81659190143857, 96: 70.98279905783664, 192: 85.94748775757752
Run 3: 48: 52.48477917290354, 96: 70.65309736868004, 192: 86.55091190492952
Run 4: 48: 50.45592799124779, 96: 70.74955582463896, 192: 85.8390177147729
Run 5: 48: 51.930386620682555, 96: 70.62675320095829, 192: 86.70764525989433
Run 6: 48: 51.380912382107276, 96: 69.2754673988788, 192: 86.40076414331213
Run 7: 48: 52.665245282185545, 96: 70.04648675779244, 192: 86.96229337949258
Run 8: 48: 52.18597985242868, 96: 70.35738107291135, 192: 86.62394316475114
Run 9: 48: 52.321574076429584, 96: 70.74446428363973, 192: 86.89686314471356
Run 10: 48: 52.177355232176836, 96: 70.62157656000805, 192: 86.8822699924568
Average runtime across all runs: 48: 51.94, 96: 70.43, 192: 86.47


In [38]:
with open(str(parent_dir) + '/Results_Evaluation/AutoGluonTS_NAS_Res_RMSE_Avg_10_Runs.pkl', 'rb') as f:
        r = pickle.load(f)
runtimes = []
for seq_len in [48, 96, 192]: 
    runtime = r['NASTrain_Energy_Prices_Data'][seq_len]['Training_Testing_Time']
    runtimes.append(runtime)
print(f"Average runtime: 48: {runtimes[0]:.2f}, 96: {runtimes[1]:.2f}, 192: {runtimes[2]:.2f}")

Average runtime: 48: 51.94, 96: 70.43, 192: 86.47


In [64]:
BASE = '/home/RDC/khankiss/H/AutoGluon_Bench'
SEQ_LENS = [48, 96, 192]

def runtime_table_autogluon(base, n_runs=10, seq_lens=SEQ_LENS):
    rows = []
    for seq_len in seq_lens:
        for run in range(1, n_runs + 1):
            path = f'{base}/AutoGluonTS_NAS_Res_RMSE_{run}.pkl'
            with open(path, 'rb') as f:
                r = pickle.load(f)
            rows.append({'tree': 'AutoGluon', 'horizon': seq_len, 'run': run,
                            'runtime': r['NASTrain_Energy_Prices_Data'][seq_len]['Training_Testing_Time']})
    return pd.DataFrame(rows)

rt = runtime_table_autogluon(BASE)
rt.to_pickle('runtime_autogluon.pkl')

### Runtime NAS Tree

In [63]:
BASE = '/home/RDC/khankiss/H/Results_Point_Forecasts_new'
SEQ_LENS = [48, 96, 192]

def runtime_table(base, prefix, n_trees=10, n_runs=10, seq_lens=SEQ_LENS):
    rows = []
    for tree in range(n_trees):
        for seq_len in seq_lens:
            for run in range(1, n_runs + 1):
                path = f'{base}/{prefix}_{tree}_EU_Energy_Prices_Time_Series_{seq_len}_{run}.pkl'
                with open(path, 'rb') as f:
                    r = pickle.load(f)
                rows.append({'tree': tree, 'horizon': seq_len, 'run': run,
                             'runtime': r['Training_Testing_Time']})
    return pd.DataFrame(rows)

rt = runtime_table(BASE, 'tree')
rt.to_pickle('runtime_tree_ensemble.pkl')

with open(parent_dir+'/Src/point_metrics_all_benchmarks_long.pkl', 'rb') as f:
    all = pickle.load(f)
clean = all[(all.arm == 'tree') & (all.tree != 'ensemble')]

surviving = set(map(tuple, clean[['horizon', 'tree', 'run']].drop_duplicates().values))
rt['keep'] = [tuple(x) in surviving for x in rt[['horizon', 'tree', 'run']].values]
rt_clean = rt[rt.keep]
rt
per_run = rt_clean.groupby(['horizon', 'run']).runtime.agg(['sum', 'max', 'size'])
print(per_run.groupby('horizon').mean().round(2))   # mean over runs of sum / max / member count

             sum     max  size
horizon                       
48       1197.29  144.45  10.0
96       1497.88  172.97  10.0
192      2099.51  249.93  10.0


In [46]:
import numpy as np
all_runtimes = []
for tree in range(10):
    runtimes = []
    for seq_len in [48, 96, 192]: 
        with open(f'/home/RDC/khankiss/H/Results_Point_Forecasts_new/tree_{tree}_EU_Energy_Prices_Time_Series_{seq_len}_Avg_10_Runs.pkl', 'rb') as f:
            r = pickle.load(f)
        runtime = r['Training_Testing_Time']
        runtimes.append(runtime)
    all_runtimes.append(runtimes)
    print(f"Tree {tree}: 48: {runtimes[0]:.2f}, 96: {runtimes[1]:.2f}, 192: {runtimes[2]:.2f}")
print(f"Average runtime across all trees: 48: {np.max([r[0] for r in all_runtimes]).round(2)}, 96: {np.max([r[1] for r in all_runtimes]).round(2)}, 192: {np.max([r[2] for r in all_runtimes]).round(2)}")

Tree 0: 48: 98.63, 96: 112.62, 192: 201.36
Tree 1: 48: 143.27, 96: 165.60, 192: 211.35
Tree 2: 48: 99.19, 96: 146.80, 192: 219.75
Tree 3: 48: 138.52, 96: 167.47, 192: 223.31
Tree 4: 48: 123.98, 96: 143.62, 192: 153.50
Tree 5: 48: 120.94, 96: 172.97, 192: 239.80
Tree 6: 48: 99.02, 96: 129.25, 192: 224.61
Tree 7: 48: 105.01, 96: 150.68, 192: 248.96
Tree 8: 48: 142.75, 96: 164.29, 192: 199.54
Tree 9: 48: 125.99, 96: 144.57, 192: 177.33
Average runtime across all trees: 48: 143.27, 96: 172.97, 192: 248.96


### Runtime NAS Chain

In [60]:
import pickle
import numpy as np
import pandas as pd

BASE = '/home/RDC/khankiss/H/Results_Chain'
SEQ_LENS = [48, 96, 192]

def runtime_table(base, prefix, n_trees=10, n_runs=10, seq_lens=SEQ_LENS):
    rows = []
    for tree in range(n_trees):
        for seq_len in seq_lens:
            for run in range(1, n_runs + 1):
                path = f'{base}/{prefix}_{tree}_EU_Energy_Prices_Time_Series_{seq_len}_{run}.pkl'
                with open(path, 'rb') as f:
                    r = pickle.load(f)
                rows.append({'tree': tree, 'horizon': seq_len, 'run': run,
                             'runtime': r['Training_Testing_Time']})
    return pd.DataFrame(rows)

rt = runtime_table(BASE, 'chain')

rt.to_pickle('runtime_chain_ensemble.pkl')
with open(parent_dir+'/Src/point_metrics_all_benchmarks_long.pkl', 'rb') as f:
    all = pickle.load(f)
clean = all[(all.arm == 'chain') & (all.tree != 'ensemble')]

surviving = set(map(tuple, clean[['horizon', 'tree', 'run']].drop_duplicates().values))
rt['keep'] = [tuple(x) in surviving for x in rt[['horizon', 'tree', 'run']].values]
rt_clean = rt[rt.keep]
rt_clean.to_pickle('runtime_chain_ensemble_clean.pkl')

In [ ]:
per_run = rt_clean.groupby(['horizon', 'run']).runtime.agg(['sum', 'max', 'size'])
print(per_run.groupby('horizon').mean().round(2))   # mean over runs of sum / max / member count

            sum     max  size
horizon                      
48       243.96   57.35   6.9
96       250.48   63.30   6.0
192      430.52  119.76   5.9


### Runtime Tree ensemble - without scheduler and path proba lr multiplier = 1.0

In [9]:
BASE = '/home/RDC/khankiss/H/Results_Tree_noscheduler'
SEQ_LENS = [48, 96, 192]

def runtime_table(base, prefix, n_trees=10, n_runs=10, seq_lens=SEQ_LENS):
    rows = []
    for tree in range(n_trees):
        for seq_len in seq_lens:
            for run in range(1, n_runs + 1):
                path = f'{base}/{prefix}_{tree}_EU_Energy_Prices_Time_Series_{seq_len}_{run}.pkl'
                with open(path, 'rb') as f:
                    r = pickle.load(f)
                rows.append({'tree': tree, 'horizon': seq_len, 'run': run,
                             'runtime': r['Training_Testing_Time']})
    return pd.DataFrame(rows)

rt = runtime_table(BASE, 'tree')

with open(parent_dir+'/Src/point_metrics_all_benchmarks_long.pkl', 'rb') as f:
    all = pickle.load(f)
clean = all[(all.arm == 'tree') & (all.tree != 'ensemble')]

surviving = set(map(tuple, clean[['horizon', 'tree', 'run']].drop_duplicates().values))
rt['keep'] = [tuple(x) in surviving for x in rt[['horizon', 'tree', 'run']].values]
rt_clean = rt[rt.keep]

per_run = rt_clean.groupby(['horizon', 'run']).runtime.agg(['sum', 'max', 'size'])
print(per_run.groupby('horizon').mean().round(2))   # mean over runs of sum / max / member count

             sum     max  size
horizon                       
48        992.86  116.11  10.0
96       1366.08  150.45  10.0
192      2070.79  248.41  10.0


### Pretrained Models Runtime

In [61]:
BASE = '/home/RDC/khankiss/H/Pretrained_Benchmark'
SEQ_LENS = [48, 96, 192]

def runtime_table_pretrained(base, models = ['GPT4TS', 'Sundial', 'TimeMoe', 'TimesFM'], n_runs=10, seq_lens=SEQ_LENS):
    rows = []
    for model in models:
        for seq_len in seq_lens:
            if model in ['TimeMoe', 'TimesFM']:
                path = f'{base}/{model}_EU_Energy_Prices_Time_Series_{seq_len}_1.pkl'
                with open(path, 'rb') as f:
                    r = pickle.load(f)
                rows.append({'tree': model, 'horizon': seq_len, 'run': int(1),
                            'runtime': r['Training_Testing_Time']})
            else:
                for run in range(1, n_runs + 1):
                    path = f'{base}/{model}_EU_Energy_Prices_Time_Series_{seq_len}_{run}.pkl'
                    with open(path, 'rb') as f:
                        r = pickle.load(f)
                    rows.append({'tree': model, 'horizon': seq_len, 'run': run,
                                'runtime': r['Training_Testing_Time']})
    return pd.DataFrame(rows)

rt = runtime_table_pretrained(BASE)
rt.to_pickle('runtime_pretrained_models.pkl')
per_run = rt.groupby(['horizon', 'run']).runtime.agg(['sum', 'max', 'size'])
print(per_run.groupby('horizon').mean().round(2))   # mean over runs of sum / max / member count

             sum      max  size
horizon                        
48       1740.75  1447.11   2.2
96       1782.44  1394.40   2.2
192      1879.53  1415.78   2.2


In [42]:
rt.groupby(['tree', 'horizon']).runtime.agg(['sum','max', 'mean']).round(2)

sum      max     mean
tree    horizon                            
GPT4TS  48        1368.61   142.88   136.86
        96        2301.59   249.31   230.16
        192       3129.92   350.22   312.99
Sundial 48       11398.19  1433.57  1139.82
        96       11451.28  1442.78  1145.13
        192      10500.03  1316.46  1050.00
TimeMoe 48        4505.10  4505.10  4505.10
        96        3933.30  3933.30  3933.30
        192       4972.76  4972.76  4972.76
TimesFM 48         135.60   135.60   135.60
        96         138.21   138.21   138.21
        192        192.55   192.55   192.55

In [47]:
print(rt.groupby(['tree', 'horizon']).runtime.agg(['sum','max', 'mean', 'size']).round(2))

                      sum      max     mean  size
tree    horizon                                  
GPT4TS  48        1368.61   142.88   136.86    10
        96        2301.59   249.31   230.16    10
        192       3129.92   350.22   312.99    10
Sundial 48       11398.19  1433.57  1139.82    10
        96       11451.28  1442.78  1145.13    10
        192      10500.03  1316.46  1050.00    10
TimeMoe 48        4505.10  4505.10  4505.10     1
        96        3933.30  3933.30  3933.30     1
        192       4972.76  4972.76  4972.76     1
TimesFM 48         135.60   135.60   135.60     1
        96         138.21   138.21   138.21     1
        192        192.55   192.55   192.55     1


### Multivariate and Univariate Models Runtime

In [62]:
BASE = '/home/RDC/khankiss/H/Benchmark_Multivariate'
SEQ_LENS = [48, 96, 192]

def runtime_table_multivariate(base, models = ['Basisformer', 'Dlinear', 'PatchTST', 'TSMixer', 'MTSMixer'], n_runs=10, seq_lens=SEQ_LENS):
    rows = []
    for model in models:
        for seq_len in seq_lens:
            for run in range(1, n_runs + 1):
                path = f'{base}/{model}_EU_Energy_Prices_Time_Series_{seq_len}_{run}.pkl'
                with open(path, 'rb') as f:
                    r = pickle.load(f)
                rows.append({'tree': model, 'horizon': seq_len, 'run': run,
                            'runtime': r['Training_Testing_Time']})
    return pd.DataFrame(rows)

rt = runtime_table_multivariate(BASE)
rt.to_pickle('runtime_multivariate_models.pkl')
per_run = rt.groupby(['horizon', 'run']).runtime.agg(['sum', 'max', 'size'])
print(per_run.groupby('horizon').mean().round(2))   # mean over runs of sum / max / member count

            sum     max  size
horizon                      
48       560.83  210.81   5.0
96       654.42  255.15   5.0
192      670.61  367.80   5.0


In [44]:
rt.groupby(['tree', 'horizon']).runtime.agg(['sum','max', 'mean']).round(2)

sum     max    mean
tree        horizon                         
Basisformer 48       2108.05  253.08  210.81
            96       2086.50  288.22  208.65
            192      1871.31  217.09  187.13
Dlinear     48        598.44   68.97   59.84
            96        619.42   71.16   61.94
            192       359.76   40.43   35.98
MTSMixer    48        627.99   72.06   62.80
            96        690.70   82.37   69.07
            192       410.01   48.71   41.00
PatchTST    48       1687.33  181.68  168.73
            96       2523.37  271.41  252.34
            192      3678.03  394.09  367.80
TSMixer     48        586.51   65.60   58.65
            96        624.19   70.07   62.42
            192       387.03   41.57   38.70

## Ablation: no scheduler and no gate lr multiplier

In [13]:
import point_metrics_v2 as pm
import pandas as pd

trees = range(10)
seq_lens = [48, 96, 192]

# one tree, one run, to check paths and shapes before the full pass
d = pm.load_run(parent_dir='/home/RDC/khankiss/H', folder_name='Results_Tree_noscheduler', tree_idx=0, seq_len=48, run=1)
print(len(d), 'splits')
print(next(iter(d.values()))[0].shape)     # expect (n_sequences, 48, 19)

# all trees
# df = pm.tree_table(trees, seq_lens, parent_dir='/home/RDC/khankiss/H', folder_name='Results_Tree_noscheduler')
print(pm.summarise(df).round(4))

# ensemble, formed within each run
ens = pm.ensemble_table(trees, seq_lens, parent_dir='/home/RDC/khankiss/H', folder_name='Results_Tree_noscheduler')
print(pm.summarise(ens).round(4))

both = pd.concat([df, ens], ignore_index=True)
# both.to_pickle('point_metrics_long_noscheduler.pkl')

keep = [f'Train_Test{i}' for i in range(1, 16)]
ens = ens[ens.split.isin(keep)]
df  = df[df.split.isin(keep)]
order = sorted(ens.split.unique(), key=lambda s: int(s.replace('Train_Test', '')))
tab = ens.groupby(['horizon', 'split']).RMSE.mean().unstack()[order]
# tab

pm.summarise(pd.concat([df, ens])).round(4)

17 splits
(308, 48, 19)
                RMSE     MSE     MAE   MedAE  RMSE_sd_run  MSE_sd_run  \
tree horizon                                                            
0    48       0.8619  0.8046  0.5943  0.4274       0.0051      0.0109   
     96       0.9154  0.8757  0.6504  0.4878       0.0087      0.0170   
     192      1.0094  1.1631  0.7361  0.5638       0.0377      0.1546   
1    48       0.8702  0.8187  0.5987  0.4305       0.0030      0.0067   
     96       0.9006  0.8476  0.6311  0.4650       0.0048      0.0099   
     192      0.9467  0.9752  0.6741  0.5080       0.0209      0.0405   
2    48       0.8685  0.8182  0.5952  0.4284       0.0104      0.0215   
     96       0.8866  0.8236  0.6209  0.4593       0.0032      0.0060   
     192      0.9406  0.9603  0.6682  0.5058       0.0374      0.0817   
3    48       0.8853  0.8488  0.6110  0.4430       0.0182      0.0370   
     96       0.9141  0.8767  0.6467  0.4810       0.0140      0.0253   
     192      0.9601  1.006

RMSE     MSE     MAE   MedAE  RMSE_sd_run  MSE_sd_run  \
tree     horizon                                                            
0        48       0.8923  0.8574  0.6133  0.4403       0.0052      0.0118   
         96       0.9221  0.8901  0.6537  0.4893       0.0096      0.0193   
         192      1.0094  1.1631  0.7361  0.5638       0.0377      0.1546   
1        48       0.9021  0.8738  0.6196  0.4458       0.0031      0.0072   
         96       0.9016  0.8516  0.6293  0.4616       0.0054      0.0111   
         192      0.9467  0.9752  0.6741  0.5080       0.0209      0.0405   
2        48       0.9002  0.8733  0.6154  0.4427       0.0108      0.0231   
         96       0.8883  0.8287  0.6203  0.4584       0.0036      0.0070   
         192      0.9406  0.9603  0.6682  0.5058       0.0374      0.0817   
3        48       0.9173  0.9053  0.6325  0.4598       0.0184      0.0390   
         96       0.9144  0.8791  0.6447  0.4779       0.0137      0.0245   
         192      0.9601  1.0063  0.6854  0.5169       0.0127      0.0357   
4        48       0.9138  0.9021  0.6260  0.4470       0.0122      0.0280   
         96       0.9017  0.8522  0.6299  0.4611       0.0118      0.0242   
         192      0.9292  0.9358  0.6583  0.4939       0.0160      0.0347   
5        48       0.9037  0.8770  0.6176  0.4426       0.0036      0.0082   
         96       0.9133  0.8778  0.6417  0.4738       0.0081      0.0146   
         192      0.9743  1.0279  0.6937  0.5232       0.0091      0.0250   
6        48       0.9046  0.8779  0.6206  0.4494       0.0077      0.0168   
         96       0.9211  0.8894  0.6548  0.4952       0.0097      0.0182   
         192      0.9814  1.0287  0.7078  0.5440       0.0192      0.0451   
7        48       0.8983  0.8680  0.6171  0.4454       0.0132      0.0261   
         96       0.9153  0.8801  0.6440  0.4791       0.0164      0.0316   
         192      0.9760  1.0261  0.6938  0.5255       0.0172      0.0433   
8        48       0.9071  0.8870  0.6240  0.4446       0.0028      0.0063   
         96       0.8951  0.8358  0.6221  0.4534       0.0037      0.0065   
         192      0.9108  0.9085  0.6363  0.4658       0.0065      0.0204   
9        48       0.9124  0.8947  0.6314  0.4608       0.0222      0.0456   
         96       0.9229  0.8975  0.6513  0.4829       0.0204      0.0434   
         192      0.9599  1.0159  0.6905  0.5258       0.0167      0.0391   
ensemble 48       0.8743  0.8203  0.5912  0.4197       0.0008      0.0015   
         96       0.8757  0.8050  0.6082  0.4444       0.0016      0.0029   
         192      0.9003  0.8730  0.6297  0.4659       0.0047      0.0100   

                  MAE_sd_run  MedAE_sd_run  RMSE_sd_split  MSE_sd_split  \
tree     horizon                                                          
0        48           0.0047        0.0043         0.2558        0.4975   
         96           0.0085        0.0081         0.2025        0.3932   
         192          0.0355        0.0338         0.3707        0.9899   
1        48           0.0033        0.0031         0.2535        0.4969   
         96           0.0064        0.0071         0.2028        0.3948   
         192          0.0177        0.0192         0.2848        0.6003   
2        48           0.0090        0.0070         0.2585        0.5078   
         96           0.0028        0.0029         0.2058        0.4031   
         192          0.0362        0.0365         0.2768        0.5780   
3        48           0.0184        0.0206         0.2601        0.5239   
         96           0.0118        0.0111         0.2110        0.4193   
         192          0.0127        0.0162         0.2920        0.6362   
4        48           0.0096        0.0091         0.2663        0.5365   
         96           0.0127        0.0118         0.2033        0.3923   
         192          0.0150        0.0169         0.2746        0.5722   
5        48           0.0026        0.0025       

In [14]:
both.to_pickle('point_metrics_long_noscheduler.pkl')

In [17]:
print(pm.summarise(ens).round(4))

                    RMSE     MSE     MAE   MedAE  RMSE_sd_run  MSE_sd_run  \
tree     horizon                                                            
ensemble 48       0.8743  0.8203  0.5912  0.4197       0.0008      0.0015   
         96       0.8757  0.8050  0.6082  0.4444       0.0016      0.0029   
         192      0.9003  0.8730  0.6297  0.4659       0.0047      0.0100   

                  MAE_sd_run  MedAE_sd_run  RMSE_sd_split  MSE_sd_split  \
tree     horizon                                                          
ensemble 48           0.0014        0.0022         0.2445        0.4634   
         96           0.0018        0.0023         0.2021        0.3873   
         192          0.0048        0.0046         0.2580        0.5148   

                  MAE_sd_split  MedAE_sd_split  
tree     horizon                                
ensemble 48             0.1196          0.0817  
         96             0.1122          0.0868  
         192            0.1633          0.

### Ablation: no scheduler only [with gate multiplier 0.4 (NAS setup)]

In [18]:
trees = range(10)
seq_lens = [48, 96, 192]

# one tree, one run, to check paths and shapes before the full pass
d = pm.load_run(parent_dir='/home/RDC/khankiss/H', folder_name='Results_Tree_noscheduler_gatelr', tree_idx=0, seq_len=48, run=1)
print(len(d), 'splits')
print(next(iter(d.values()))[0].shape)     # expect (n_sequences, 48, 19)

# all trees
df = pm.tree_table(trees, seq_lens, parent_dir='/home/RDC/khankiss/H', folder_name='Results_Tree_noscheduler_gatelr')
print(pm.summarise(df).round(4))

# ensemble, formed within each run
ens = pm.ensemble_table(trees, seq_lens, parent_dir='/home/RDC/khankiss/H', folder_name='Results_Tree_noscheduler_gatelr')
print(pm.summarise(ens).round(4))

both = pd.concat([df, ens], ignore_index=True)
both.to_pickle('point_metrics_long_noscheduler.pkl')

keep = [f'Train_Test{i}' for i in range(1, 16)]
ens = ens[ens.split.isin(keep)]
df  = df[df.split.isin(keep)]
order = sorted(ens.split.unique(), key=lambda s: int(s.replace('Train_Test', '')))
tab = ens.groupby(['horizon', 'split']).RMSE.mean().unstack()[order]
# tab

pm.summarise(pd.concat([df, ens])).round(4)

17 splits
(308, 48, 19)
                RMSE     MSE     MAE   MedAE  RMSE_sd_run  MSE_sd_run  \
tree horizon                                                            
0    48       0.8640  0.8081  0.5955  0.4288       0.0054      0.0114   
     96       0.9250  0.8962  0.6590  0.4969       0.0208      0.0450   
     192      0.9899  1.0921  0.7208  0.5551       0.0363      0.1368   
1    48       0.8719  0.8245  0.5994  0.4304       0.0088      0.0205   
     96       0.8972  0.8407  0.6294  0.4648       0.0068      0.0140   
     192      0.9557  0.9953  0.6854  0.5229       0.0415      0.0881   
2    48       0.8717  0.8246  0.5977  0.4287       0.0066      0.0143   
     96       0.9037  0.8546  0.6330  0.4656       0.0128      0.0289   
     192      0.9537  0.9850  0.6839  0.5231       0.0397      0.0823   
3    48       0.8817  0.8427  0.6106  0.4434       0.0197      0.0455   
     96       0.9145  0.8780  0.6475  0.4827       0.0193      0.0375   
     192      0.9642  1.030

RMSE     MSE     MAE   MedAE  RMSE_sd_run  MSE_sd_run  \
tree     horizon                                                            
0        48       0.8954  0.8623  0.6154  0.4429       0.0059      0.0128   
         96       0.9319  0.9114  0.6624  0.4986       0.0202      0.0453   
         192      0.9899  1.0921  0.7208  0.5551       0.0363      0.1368   
1        48       0.9040  0.8804  0.6201  0.4452       0.0098      0.0229   
         96       0.8982  0.8445  0.6276  0.4617       0.0071      0.0146   
         192      0.9557  0.9953  0.6854  0.5229       0.0415      0.0881   
2        48       0.9042  0.8811  0.6183  0.4429       0.0077      0.0165   
         96       0.9072  0.8628  0.6333  0.4644       0.0148      0.0331   
         192      0.9537  0.9850  0.6839  0.5231       0.0397      0.0823   
3        48       0.9139  0.8994  0.6322  0.4599       0.0209      0.0499   
         96       0.9161  0.8830  0.6467  0.4812       0.0196      0.0382   
         192      0.9642  1.0303  0.6869  0.5163       0.0225      0.0799   
4        48       0.9129  0.9022  0.6253  0.4458       0.0141      0.0352   
         96       0.9057  0.8600  0.6354  0.4680       0.0166      0.0309   
         192      0.9424  0.9777  0.6641  0.4927       0.0192      0.0578   
5        48       0.9047  0.8788  0.6196  0.4455       0.0070      0.0150   
         96       0.9142  0.8777  0.6423  0.4747       0.0084      0.0144   
         192      0.9629  1.0051  0.6864  0.5202       0.0152      0.0404   
6        48       0.9002  0.8662  0.6160  0.4457       0.0038      0.0076   
         96       0.9172  0.8824  0.6507  0.4912       0.0155      0.0275   
         192      0.9773  1.0228  0.7032  0.5392       0.0181      0.0426   
7        48       0.8911  0.8529  0.6112  0.4385       0.0062      0.0128   
         96       0.9052  0.8586  0.6367  0.4739       0.0183      0.0348   
         192      0.9742  1.0330  0.6931  0.5240       0.0216      0.0605   
8        48       0.9069  0.8874  0.6235  0.4442       0.0025      0.0070   
         96       0.8971  0.8394  0.6240  0.4551       0.0023      0.0036   
         192      0.9089  0.9018  0.6351  0.4652       0.0093      0.0246   
9        48       0.9148  0.9040  0.6307  0.4561       0.0155      0.0365   
         96       0.9203  0.8904  0.6461  0.4784       0.0114      0.0210   
         192      0.9745  1.0857  0.6945  0.5222       0.0337      0.1441   
ensemble 48       0.8743  0.8204  0.5916  0.4195       0.0020      0.0044   
         96       0.8775  0.8077  0.6102  0.4466       0.0036      0.0060   
         192      0.9049  0.8829  0.6338  0.4699       0.0096      0.0220   

                  MAE_sd_run  MedAE_sd_run  RMSE_sd_split  MSE_sd_split  \
tree     horizon                                                          
0        48           0.0045        0.0043         0.2542        0.4990   
         96           0.0166        0.0165         0.2049        0.4060   
         192          0.0318        0.0251         0.3257        0.7898   
1        48           0.0082        0.0076         0.2592        0.5143   
         96           0.0082        0.0100         0.1998        0.3876   
         192          0.0375        0.0376         0.2875        0.6089   
2        48           0.0066        0.0061         0.2599        0.5182   
         96           0.0157        0.0175         0.2030        0.3933   
         192          0.0385        0.0396         0.2736        0.5724   
3        48           0.0189        0.0188         0.2596        0.5179   
         96           0.0133        0.0069         0.2095        0.4161   
         192          0.0134        0.0079         0.3161        0.7317   
4        48           0.0114        0.0114         0.2693        0.5464   
         96           0.0147        0.0132         0.2032        0.3965   
         192          0.0135        0.0122         0.3020        0.6666   
5        48           0.0071        0.0082       

In [20]:
BASE = '/home/RDC/khankiss/H/Results_Tree_noscheduler_gatelr'
SEQ_LENS = [48, 96, 192]

def runtime_table(base, prefix, n_trees=10, n_runs=10, seq_lens=SEQ_LENS):
    rows = []
    for tree in range(n_trees):
        for seq_len in seq_lens:
            for run in range(1, n_runs + 1):
                path = f'{base}/{prefix}_{tree}_EU_Energy_Prices_Time_Series_{seq_len}_{run}.pkl'
                with open(path, 'rb') as f:
                    r = pickle.load(f)
                rows.append({'tree': tree, 'horizon': seq_len, 'run': run,
                             'runtime': r['Training_Testing_Time']})
    return pd.DataFrame(rows)

rt = runtime_table(BASE, 'tree')

with open(parent_dir+'/Src/point_metrics_all_benchmarks_long.pkl', 'rb') as f:
    all = pickle.load(f)
clean = all[(all.arm == 'tree') & (all.tree != 'ensemble')]

surviving = set(map(tuple, clean[['horizon', 'tree', 'run']].drop_duplicates().values))
rt['keep'] = [tuple(x) in surviving for x in rt[['horizon', 'tree', 'run']].values]
rt_clean = rt[rt.keep]

per_run = rt_clean.groupby(['horizon', 'run']).runtime.agg(['sum', 'max', 'size'])
print(per_run.groupby('horizon').mean().round(2))   # mean over runs of sum / max / member count

             sum     max  size
horizon                       
48        980.07  115.70  10.0
96       1345.80  149.23  10.0
192      1996.73  234.89  10.0


## Parameter Comparison

Produce Parameter count file stored .pkl by running [NAS_tree_params.py](../Benchmark_Models/NAS_tree_params.py)

In [6]:
import pickle
from pathlib import Path
parent_dir = str(Path.cwd().parent)
with open(parent_dir+'/Benchmark_Models/tree_NAS_params.pkl', 'rb') as f:
    tree_params = pickle.load(f)
with open(parent_dir+'/Benchmark_Models/chain_NAS_params.pkl', 'rb') as f:
    chain_params = pickle.load(f)

import pandas as pd
print(pd.concat([tree_params, chain_params], ignore_index=True).groupby(['horizon', 'arm',  'tree']).sum().unstack())

               params                                                        \
tree                0        1        2        3        4        5        6   
horizon arm                                                                   
48      chain  228686    94320  1077034   242972   242972   262376    17836   
        tree    26958   595106   122981   556175   131897   453624   296907   
96      chain  403358   206016  2127946   494348   494348   513752    59596   
        tree    68718  1172978   165845  1097231   268121   742152   569163   
192     chain  794174   498528  4257418  1038572  1038572  1057976   226060   
        tree   235182  2370194   333173  2262287   582041  1389096  1196619   

                                         
tree                 7        8       9  
horizon arm                              
48      chain  1557355   600603   40264  
        tree    226493   333989  181775  
96      chain  2786395  1211499   68104  
        tree    322397   641429  377

## TOPSIS

Topsis related functions

In [119]:
import numpy as np
import pandas as pd

ERRORS = ['RMSE', 'MSE', 'MAE', 'MedAE']
RUNTIME = 'runtime'
SCHEMES = [round(0.1 * i, 1) for i in range(9, 0, -1)]


def _prepare(df, criteria, log_cols, normalise):
    X = df[criteria].astype(float).copy()
    for c in log_cols or []:
        if c in X:
            X[c] = np.log10(X[c])

    if normalise == 'minmax':
        rng = (X.max() - X.min()).replace(0.0, 1.0)
        return (X - X.min()) / rng
    if normalise == 'vector':
        norm = np.sqrt((X ** 2).sum(axis=0)).replace(0.0, 1.0)
        return X / norm
    raise ValueError(f"normalise must be 'minmax' or 'vector', got {normalise!r}")


def topsis(N, weights):
    """TOPSIS on an already-normalised frame. All criteria treated as costs."""
    V = N * pd.Series(weights)
    best, worst = V.min(), V.max()                       # costs: min is ideal
    d_best = np.sqrt(((V - best) ** 2).sum(axis=1))
    d_worst = np.sqrt(((V - worst) ** 2).sum(axis=1))
    denom = (d_best + d_worst).replace(0.0, 1.0)
    return d_worst / denom


def topsis_table(df, errors=ERRORS, runtime=RUNTIME, schemes=SCHEMES,
                 normalise='minmax', log_cols=(RUNTIME,)):
    """-> (closeness per scheme, rank per scheme with Avg_Rank appended)."""
    criteria = list(errors) + [runtime]
    N = _prepare(df, criteria, log_cols, normalise)

    scores = {}
    for w_acc in schemes:
        weights = {e: w_acc / len(errors) for e in errors}
        weights[runtime] = round(1.0 - w_acc, 1)
        scores[f'[{w_acc}, {round(1 - w_acc, 1)}]'] = topsis(N, weights)

    scores = pd.DataFrame(scores)
    ranks = scores.rank(ascending=False).astype(int)
    ranks['Avg_Rank'] = ranks.mean(axis=1).round(3)
    return scores, ranks.sort_values('Avg_Rank')


def diagnose(df, errors=ERRORS, runtime=RUNTIME, log_cols=(RUNTIME,)):
    """Per-criterion spread under each scheme, and runtime's weight at [0.9, 0.1].

    Report this to justify the normalisation rather than asserting it.
    """
    criteria = list(errors) + [runtime]
    rows = {'raw_ratio': (df[criteria].max() / df[criteria].min()).round(1)}

    for name, logs in [('vector', ()), ('minmax', ()), ('log+minmax', log_cols)]:
        how = 'vector' if name == 'vector' else 'minmax'
        N = _prepare(df, criteria, logs, how)
        rows[f'spread_{name}'] = (N.max() - N.min()).round(4)

    out = pd.DataFrame(rows)

    # influence at the most accuracy-favouring scheme
    print(out.to_string())
    print('\ninfluence at [0.9 accuracy, 0.1 runtime]:')
    for name, logs in [('vector', ()), ('minmax', ()), ('log+minmax', log_cols)]:
        sp = out[f'spread_{name}']
        err = np.sqrt(((sp[errors] * 0.225) ** 2).sum())
        rt = sp[runtime] * 0.1
        print(f'  {name:12s} errors {err:.4f}   runtime {rt:.4f}   '
              f'runtime/errors {rt / err:.2f}x')
    return out

Aggregating Metrics

In [55]:
with open('point_metrics_all_benchmarks_long.pkl', 'rb') as f:
    all_results = pickle.load(f)
keep = [f'Train_Test{i}' for i in range(1, 16)]
full = all_results[all_results.split.isin(keep)]
print(full.split.unique())
full = pm.summarise(full, by=('horizon','arm','tree'))

main = full.loc[full.index.get_level_values('tree').isin(
    ['ensemble', 'AutoGluon', 'TimeMoe', 'TimesFM', 'Sundial',
     'GPT4TS', 'Basisformer', 'TSMixer', 'MTSMixer', 'Dlinear', 'PatchTST']
)]
errs = main[['RMSE', 'MSE', 'MAE', 'MedAE']]
errs

['Train_Test1' 'Train_Test2' 'Train_Test3' 'Train_Test4' 'Train_Test5'
 'Train_Test6' 'Train_Test7' 'Train_Test8' 'Train_Test9' 'Train_Test10'
 'Train_Test11' 'Train_Test12' 'Train_Test13' 'Train_Test14'
 'Train_Test15']


RMSE       MSE       MAE     MedAE
horizon arm          tree                                               
48      autogluon    AutoGluon    1.158846  1.511316  0.750734  0.474656
        chain        ensemble     0.900491  0.915962  0.586950  0.400922
        multivariate Basisformer  0.957869  0.987923  0.646294  0.433324
                     Dlinear      1.015891  1.129297  0.703911  0.513178
                     MTSMixer     1.125449  1.379636  0.776401  0.570570
                     PatchTST     0.937655  0.941067  0.626539  0.408110
                     TSMixer      0.923117  0.917634  0.641487  0.469594
        pretrained   GPT4TS       1.207184  1.576315  0.842931  0.593191
                     Sundial      0.992735  1.055733  0.657389  0.415369
                     TimeMoe      0.940182  0.958846  0.608529  0.395198
                     TimesFM      1.011444  1.092038  0.656469  0.404177
        tree         ensemble     0.882415  0.837232  0.598952  0.427993
96      autogluon    AutoGluon    1.171148  1.460221  0.770949  0.501729
        chain        ensemble     0.870325  0.788013  0.595155  0.424718
        multivariate Basisformer  0.939998  0.914666  0.652725  0.467190
                     Dlinear      0.956089  0.967270  0.678691  0.510204
                     MTSMixer     1.066385  1.200584  0.761264  0.579437
                     PatchTST     0.921498  0.881609  0.632605  0.441626
                     TSMixer      0.914950  0.880485  0.647029  0.485890
        pretrained   GPT4TS       1.159384  1.398572  0.821454  0.598895
                     Sundial      0.952719  0.952120  0.642964  0.434912
                     TimeMoe      0.871170  0.796858  0.588754  0.412596
                     TimesFM      0.941897  0.924841  0.633276  0.427594
        tree         ensemble     0.881076  0.814241  0.614261  0.451736
192     autogluon    AutoGluon    1.018410  1.156181  0.687873  0.457474
        chain        ensemble     1.073176  1.372721  0.732233  0.508440
        multivariate Basisformer  0.917354  0.907069  0.631876  0.446259
                     Dlinear      0.941726  0.960903  0.665749  0.498488
                     MTSMixer     1.103110  1.315240  0.782389  0.585662
                     PatchTST     0.865594  0.805006  0.587558  0.406451
                     TSMixer      0.927124  0.932867  0.659187  0.498984
        pretrained   GPT4TS       1.137417  1.359058  0.802699  0.583744
                     Sundial      0.869038  0.809952  0.582288  0.395224
                     TimeMoe      0.856349  0.775914  0.576384  0.404640
                     TimesFM      0.883991  0.832090  0.586494  0.385535
        tree         ensemble     0.891889  0.853965  0.621050  0.458162

In [73]:
merged_errs = errs.copy()
merged_errs.index = pd.MultiIndex.from_arrays([
    merged_errs.index.get_level_values('horizon'),
    merged_errs.index.get_level_values('tree').astype('str')+'_'+merged_errs.index.get_level_values('arm').astype('str')
],
names=['horizon', 'method'])
merged_errs = merged_errs.rename(index={
    "AutoGluon_autogluon":      "AutoGluon",
    "ensemble_chain":           "Chain ensemble",
    "ensemble_tree":            "Tree ensemble",
    "Basisformer_multivariate": "Basisformer",
    "Dlinear_multivariate":     "Dlinear",
    "MTSMixer_multivariate":    "MTSMixer",
    "PatchTST_multivariate":    "PatchTST",
    "TSMixer_multivariate":     "TSMixer",
    "GPT4TS_pretrained":        "GPT4TS",
    "Sundial_pretrained":       "Sundial",
    "TimeMoe_pretrained":       "TimeMoe",
    "TimesFM_pretrained":       "TimesFM"
}, level='method')
merged_errs.to_pickle('topsis_error_table.pkl')
merged_errs


RMSE       MSE       MAE     MedAE
horizon method                                                
48      AutoGluon       1.158846  1.511316  0.750734  0.474656
        Chain ensemble  0.900491  0.915962  0.586950  0.400922
        Basisformer     0.957869  0.987923  0.646294  0.433324
        Dlinear         1.015891  1.129297  0.703911  0.513178
        MTSMixer        1.125449  1.379636  0.776401  0.570570
        PatchTST        0.937655  0.941067  0.626539  0.408110
        TSMixer         0.923117  0.917634  0.641487  0.469594
        GPT4TS          1.207184  1.576315  0.842931  0.593191
        Sundial         0.992735  1.055733  0.657389  0.415369
        TimeMoe         0.940182  0.958846  0.608529  0.395198
        TimesFM         1.011444  1.092038  0.656469  0.404177
        Tree ensemble   0.882415  0.837232  0.598952  0.427993
96      AutoGluon       1.171148  1.460221  0.770949  0.501729
        Chain ensemble  0.870325  0.788013  0.595155  0.424718
        Basisformer     0.939998  0.914666  0.652725  0.467190
        Dlinear         0.956089  0.967270  0.678691  0.510204
        MTSMixer        1.066385  1.200584  0.761264  0.579437
        PatchTST        0.921498  0.881609  0.632605  0.441626
        TSMixer         0.914950  0.880485  0.647029  0.485890
        GPT4TS          1.159384  1.398572  0.821454  0.598895
        Sundial         0.952719  0.952120  0.642964  0.434912
        TimeMoe         0.871170  0.796858  0.588754  0.412596
        TimesFM         0.941897  0.924841  0.633276  0.427594
        Tree ensemble   0.881076  0.814241  0.614261  0.451736
192     AutoGluon       1.018410  1.156181  0.687873  0.457474
        Chain ensemble  1.073176  1.372721  0.732233  0.508440
        Basisformer     0.917354  0.907069  0.631876  0.446259
        Dlinear         0.941726  0.960903  0.665749  0.498488
        MTSMixer        1.103110  1.315240  0.782389  0.585662
        PatchTST        0.865594  0.805006  0.587558  0.406451
        TSMixer         0.927124  0.932867  0.659187  0.498984
        GPT4TS          1.137417  1.359058  0.802699  0.583744
        Sundial         0.869038  0.809952  0.582288  0.395224
        TimeMoe         0.856349  0.775914  0.576384  0.404640
        TimesFM         0.883991  0.832090  0.586494  0.385535
        Tree ensemble   0.891889  0.853965  0.621050  0.458162

Runtime Aggregation

In [86]:
with open('runtime_chain_ensemble_clean.pkl', 'rb') as f:
    rt_chain_clean = pickle.load(f)
with open('runtime_tree_ensemble.pkl', 'rb') as f:
    rt_tree = pickle.load(f)
with open('runtime_pretrained_models.pkl', 'rb') as f:
    rt_pretrained = pickle.load(f)
with open('runtime_multivariate_models.pkl', 'rb') as f:
    rt_mv = pickle.load(f)
with open('runtime_autogluon.pkl', 'rb') as f:
    rt_ag = pickle.load(f)

rt_ag = rt_ag.groupby(['horizon', 'tree']).runtime.agg(runtime='mean').rename_axis(index={'tree':'method'})
rt_pretrained = rt_pretrained.groupby(['horizon', 'tree']).runtime.agg(runtime='mean').rename_axis(index={'tree':'method'})
rt_mv = rt_mv.groupby(['horizon', 'tree']).runtime.agg(runtime='mean').rename_axis(index={'tree':'method'})
rt_chain_clean = rt_chain_clean.groupby(['horizon', 'run']).runtime.agg(runtime='max').groupby('horizon').runtime.agg(runtime='mean')
rt_chain_clean['method']='Chain ensemble'
rt_chain_clean = rt_chain_clean.set_index('method', append=True)
rt_tree = rt_tree.groupby(['horizon', 'run']).runtime.agg(runtime='max').groupby('horizon').runtime.agg(runtime='mean')
rt_tree['method']='Tree ensemble'
rt_tree = rt_tree.set_index('method', append=True)

In [96]:
topsis_runtime = (pd.concat([rt_tree,rt_chain_clean,rt_mv,rt_pretrained,rt_ag]).reorder_levels(['horizon', 'method']).sort_index())
topsis_runtime.to_pickle('topsis_runtime.pkl')
topsis_runtime

runtime
horizon method                     
48      AutoGluon         51.942387
        Basisformer      210.805133
        Chain ensemble    57.345624
        Dlinear           59.844455
        GPT4TS           136.860913
        MTSMixer          62.798782
        PatchTST         168.732671
        Sundial         1139.819409
        TSMixer           58.650580
        TimeMoe         4505.097099
        TimesFM          135.595232
        Tree ensemble    144.447438
96      AutoGluon         70.434131
        Basisformer      208.649952
        Chain ensemble    63.302083
        Dlinear           61.941909
        GPT4TS           230.158529
        MTSMixer          69.070195
        PatchTST         252.337013
        Sundial         1145.127792
        TSMixer           62.418733
        TimeMoe         3933.297111
        TimesFM          138.206464
        Tree ensemble    172.969456
192     AutoGluon         86.473428
        Basisformer      187.130723
        Chain ensemble   119.763963
        Dlinear           35.975520
        GPT4TS           312.991988
        MTSMixer          41.000524
        PatchTST         367.802980
        Sundial         1050.003140
        TSMixer           38.703066
        TimeMoe         4972.758940
        TimesFM          192.553182
        Tree ensemble    249.930078

In [97]:
topsis_inputs = pd.concat([merged_errs, topsis_runtime], axis=1).sort_index()

### Topsis at 48h

In [105]:
topsis_inputs_all = topsis_inputs.reset_index().set_index(['method'])
topsis_input_48 = topsis_inputs_all[topsis_inputs_all['horizon'] == 48]
topsis_input_48

,horizon,RMSE,MSE,MAE,MedAE,runtime
method,,,,,,
AutoGluon,48,1.158846,1.511316,0.750734,0.474656,51.942387
Basisformer,48,0.957869,0.987923,0.646294,0.433324,210.805133
Chain ensemble,48,0.900491,0.915962,0.586950,0.400922,57.345624
Dlinear,48,1.015891,1.129297,0.703911,0.513178,59.844455
GPT4TS,48,1.207184,1.576315,0.842931,0.593191,136.860913
MTSMixer,48,1.125449,1.379636,0.776401,0.570570,62.798782
PatchTST,48,0.937655,0.941067,0.626539,0.408110,168.732671
Sundial,48,0.992735,1.055733,0.657389,0.415369,1139.819409
TSMixer,48,0.923117,0.917634,0.641487,0.469594,58.650580


In [124]:
scores_48, ranks_48 = topsis_table(topsis_input_48)
ranks_48

,"[0.9, 0.1]","[0.8, 0.2]","[0.7, 0.3]","[0.6, 0.4]","[0.5, 0.5]","[0.4, 0.6]","[0.3, 0.7]","[0.2, 0.8]","[0.1, 0.9]",Avg_Rank
method,,,,,,,,,,
Chain ensemble,1,1,1,1,1,1,1,1,1,1.000
TSMixer,4,4,3,2,2,2,2,2,2,2.556
Tree ensemble,2,2,2,3,4,6,7,7,8,4.556
Dlinear,9,9,7,6,3,3,3,3,4,5.222
TimesFM,7,6,5,5,5,7,6,6,6,5.889
PatchTST,3,3,4,4,6,8,8,9,9,6.000
AutoGluon,10,10,8,8,7,4,4,4,3,6.444
MTSMixer,11,11,10,9,8,5,5,5,5,7.667
Basisformer,5,5,6,7,9,9,10,10,10,7.889


### Topsis at 96h

In [125]:
topsis_inputs_all = topsis_inputs.reset_index().set_index(['method'])
topsis_input_96 = topsis_inputs_all[topsis_inputs_all['horizon'] == 96]
scores_96, ranks_96 = topsis_table(topsis_input_96)
ranks_96

,"[0.9, 0.1]","[0.8, 0.2]","[0.7, 0.3]","[0.6, 0.4]","[0.5, 0.5]","[0.4, 0.6]","[0.3, 0.7]","[0.2, 0.8]","[0.1, 0.9]",Avg_Rank
method,,,,,,,,,,
Chain ensemble,1,1,1,1,1,1,1,1,1,1.000
TSMixer,6,5,3,2,2,2,2,2,2,2.889
TimesFM,5,3,4,3,4,4,6,6,6,4.556
Dlinear,9,7,6,4,3,3,3,3,3,4.556
Tree ensemble,2,2,2,5,5,7,7,7,7,4.889
MTSMixer,10,10,9,8,6,5,4,4,4,6.667
Basisformer,7,6,7,6,7,8,8,8,8,7.222
PatchTST,3,4,5,7,9,9,9,10,10,7.333
AutoGluon,11,11,11,9,8,6,5,5,5,7.889


### Topsis at 192h

In [126]:
topsis_inputs_all = topsis_inputs.reset_index().set_index(['method'])
topsis_input_192 = topsis_inputs_all[topsis_inputs_all['horizon'] == 192]
scores_192, ranks_192 = topsis_table(topsis_input_192)
ranks_192

,"[0.9, 0.1]","[0.8, 0.2]","[0.7, 0.3]","[0.6, 0.4]","[0.5, 0.5]","[0.4, 0.6]","[0.3, 0.7]","[0.2, 0.8]","[0.1, 0.9]",Avg_Rank
method,,,,,,,,,,
TSMixer,7,6,2,1,1,1,1,2,2,2.556
Dlinear,8,7,3,2,2,2,2,1,1,3.111
TimesFM,1,1,1,3,4,5,6,7,7,3.889
AutoGluon,9,9,8,5,3,3,4,4,4,5.444
Basisformer,6,5,5,4,5,7,7,6,6,5.667
Tree ensemble,5,3,6,6,7,8,8,8,8,6.556
MTSMixer,11,11,10,8,6,4,3,3,3,6.556
PatchTST,2,2,4,7,9,9,9,10,10,6.889
Chain ensemble,10,10,11,9,8,6,5,5,5,7.667


## Diabold-Mariano Test - Statistical Significance test for Forecast Comparison

In [2]:
import diabold_mariano_test as dm
import pandas as pd

def build_model_forcasts(run, seq_len, include_autogluon=False, chain_keep = [1, 3, 4, 5, 6, 8]):
    PARENT_DIR = '/home/RDC/khankiss/H'
    KEY = 'EU_Energy_Prices_Time_Series'
    FOLDERS = {
        'tree':         'Results_Point_Forecasts_new',
        'chain':        'Results_Chain',
        'multivariate': 'Benchmark_Multivariate',
        'pretrained':   'Pretrained_Benchmark',
        'autogluon':    'AutoGluon_Bench',
    }
    SEQ_LEN = seq_len
    RUN = run

    file_dir = lambda folder, prefix, run: f'{PARENT_DIR}/{folder}/{prefix}_{KEY}_{SEQ_LEN}_{run}.pkl'

    models = {
        'Tree ensemble':  dm.ensemble([dm.load(path=file_dir(FOLDERS['tree'],  f'tree_{i}', RUN),  
                                            layout='tree', 
                                            seq_len=SEQ_LEN) 
                                    
                                    for i in range(10)]),

        'Chain ensemble': dm.ensemble([dm.load(path=file_dir(FOLDERS['chain'], f'chain_{i}', RUN), 
                                            layout='chain', 
                                            seq_len=SEQ_LEN)
                                    
                                    for i in chain_keep])
        }

    for m in ['Basisformer', 'Dlinear', 'PatchTST', 'TSMixer', 'MTSMixer']:

        models[m] = dm.load(path=file_dir(FOLDERS['multivariate'], m, RUN), 
                            layout='multivariate', 
                            seq_len=SEQ_LEN)

    for m in ['Sundial', 'GPT4TS']:
        models[m] = dm.load(path=file_dir(FOLDERS['pretrained'], m, RUN), 
                            layout='pretrained', 
                            seq_len=SEQ_LEN)

    for m in ['TimeMoe', 'TimesFM']: # single run only
        models[m] = dm.load(path=file_dir(FOLDERS['pretrained'], m, run=1), 
                            layout='pretrained', 
                            seq_len=SEQ_LEN)

    if include_autogluon:
        models['AutoGluon'] = dm.load_autogluon(
                                path=f'{PARENT_DIR}/{FOLDERS["autogluon"]}/AutoGluonTS_NAS_Res_RMSE_{RUN}.pkl', 
                                seq_len=SEQ_LEN)   # costs split 1

    return models


### norm=2 (MSE)

In [39]:
SEQ_LEN = 48
STRIDE = SEQ_LEN
p_values = {}
for run in range(1,11,1):
    models = build_model_forcasts(run=run, seq_len=SEQ_LEN, chain_keep=[1, 3, 4, 5, 6, 8])
    fc = dm.stack(models=models, stride=STRIDE)
    p_values[run] = dm.dm_matrix(fc=fc, focus='Tree ensemble', norm=2).set_index('vs')['p_focus_better']

P = pd.DataFrame(p_values)
print(P.agg(['min', 'median', 'max'], axis=1).round(4))
print('runs significant at 5%:', (P < 0.05).sum(axis=1).to_dict())

splits 1-15 (15), 105 origins
splits 1-15 (15), 105 origins
splits 1-15 (15), 105 origins
splits 1-15 (15), 105 origins
splits 1-15 (15), 105 origins
splits 1-15 (15), 105 origins
splits 1-15 (15), 105 origins
splits 1-15 (15), 105 origins
splits 1-15 (15), 105 origins
splits 1-15 (15), 105 origins
                   min  median     max
vs                                    
Basisformer     0.0013  0.0029  0.0104
Chain ensemble  0.3955  0.4746  0.5667
Dlinear         0.0000  0.0000  0.0000
GPT4TS          0.0000  0.0000  0.0000
MTSMixer        0.0000  0.0000  0.0000
PatchTST        0.0054  0.0128  0.0239
Sundial         0.0001  0.0003  0.0008
TSMixer         0.0000  0.0000  0.0026
TimeMoe         0.0001  0.0004  0.0035
TimesFM         0.0000  0.0001  0.0003
runs significant at 5%: {'Basisformer': 10, 'Chain ensemble': 0, 'Dlinear': 10, 'GPT4TS': 10, 'MTSMixer': 10, 'PatchTST': 10, 'Sundial': 10, 'TSMixer': 10, 'TimeMoe': 10, 'TimesFM': 10}


In [40]:
from statsmodels.stats.multitest import multipletests

P_adj = P.apply(
    lambda col: pd.Series(
        multipletests(col.values, alpha=0.05, method="fdr_bh")[1], index=col.index
    ),
    axis=0,
)

print(P_adj.agg(['min', 'median', 'max'], axis=1).round(4))
print('runs significant at 5% after BH:', (P_adj < 0.05).sum(axis=1).to_dict())

                   min  median     max
vs                                    
Basisformer     0.0016  0.0036  0.0116
Chain ensemble  0.3955  0.4746  0.5667
Dlinear         0.0000  0.0000  0.0000
GPT4TS          0.0000  0.0000  0.0000
MTSMixer        0.0000  0.0000  0.0000
PatchTST        0.0068  0.0142  0.0266
Sundial         0.0002  0.0004  0.0011
TSMixer         0.0000  0.0000  0.0037
TimeMoe         0.0002  0.0006  0.0044
TimesFM         0.0000  0.0002  0.0005
runs significant at 5% after BH: {'Basisformer': 10, 'Chain ensemble': 0, 'Dlinear': 10, 'GPT4TS': 10, 'MTSMixer': 10, 'PatchTST': 10, 'Sundial': 10, 'TSMixer': 10, 'TimeMoe': 10, 'TimesFM': 10}


In [37]:
SEQ_LEN = 96
STRIDE = SEQ_LEN
p_values = {}
for run in range(1,11,1):
    models = build_model_forcasts(run=run, seq_len=SEQ_LEN, chain_keep=[1, 3, 4, 5, 6, 8])
    fc = dm.stack(models=models, stride=STRIDE)
    p_values[run] = dm.dm_matrix(fc=fc, focus='Tree ensemble', norm=2).set_index('vs')['p_focus_better']

P = pd.DataFrame(p_values)
print(P.agg(['min', 'median', 'max'], axis=1).round(4))
print('runs significant at 5%:', (P < 0.05).sum(axis=1).to_dict())

splits 1-15 (15), 60 origins
splits 1-15 (15), 60 origins
splits 1-15 (15), 60 origins
splits 1-15 (15), 60 origins
splits 1-15 (15), 60 origins
splits 1-15 (15), 60 origins
splits 1-15 (15), 60 origins
splits 1-15 (15), 60 origins
splits 1-15 (15), 60 origins
splits 1-15 (15), 60 origins
                   min  median     max
vs                                    
Basisformer     0.0001  0.0014  0.0292
Chain ensemble  0.2021  0.7821  0.8779
Dlinear         0.0000  0.0000  0.0000
GPT4TS          0.0000  0.0000  0.0000
MTSMixer        0.0000  0.0000  0.0000
PatchTST        0.0070  0.0163  0.0654
Sundial         0.0022  0.0054  0.0256
TSMixer         0.0000  0.0000  0.0017
TimeMoe         0.6664  0.7755  0.8902
TimesFM         0.0042  0.0104  0.0385
runs significant at 5%: {'Basisformer': 10, 'Chain ensemble': 0, 'Dlinear': 10, 'GPT4TS': 10, 'MTSMixer': 10, 'PatchTST': 8, 'Sundial': 10, 'TSMixer': 10, 'TimeMoe': 0, 'TimesFM': 10}


In [38]:
from statsmodels.stats.multitest import multipletests

P_adj = P.apply(
    lambda col: pd.Series(
        multipletests(col.values, alpha=0.05, method="fdr_bh")[1], index=col.index
    ),
    axis=0,
)

print(P_adj.agg(['min', 'median', 'max'], axis=1).round(4))
print('runs significant at 5% after BH:', (P_adj < 0.05).sum(axis=1).to_dict())

                   min  median     max
vs                                    
Basisformer     0.0002  0.0027  0.0487
Chain ensemble  0.2246  0.7850  0.8864
Dlinear         0.0000  0.0000  0.0000
GPT4TS          0.0000  0.0000  0.0000
MTSMixer        0.0000  0.0000  0.0000
PatchTST        0.0087  0.0217  0.0817
Sundial         0.0037  0.0089  0.0487
TSMixer         0.0000  0.0000  0.0042
TimeMoe         0.6664  0.8222  0.8902
TimesFM         0.0060  0.0146  0.0550
runs significant at 5% after BH: {'Basisformer': 10, 'Chain ensemble': 0, 'Dlinear': 10, 'GPT4TS': 10, 'MTSMixer': 10, 'PatchTST': 8, 'Sundial': 10, 'TSMixer': 10, 'TimeMoe': 0, 'TimesFM': 8}


In [41]:
SEQ_LEN = 192
STRIDE = SEQ_LEN
p_values = {}
for run in range(1,11,1):
    models = build_model_forcasts(run=run, seq_len=SEQ_LEN, chain_keep=[1, 3, 4, 5, 8])
    fc = dm.stack(models=models, stride=STRIDE)
    p_values[run] = dm.dm_matrix(fc=fc, focus='Tree ensemble', norm=2).set_index('vs')['p_focus_better']

P = pd.DataFrame(p_values)
print(P.agg(['min', 'median', 'max'], axis=1).round(4))
print('runs significant at 5%:', (P < 0.05).sum(axis=1).to_dict())

splits 1-15 (15), 30 origins
splits 1-15 (15), 30 origins
splits 1-15 (15), 30 origins
splits 1-15 (15), 30 origins
splits 1-15 (15), 30 origins
splits 1-15 (15), 30 origins
splits 1-15 (15), 30 origins
splits 1-15 (15), 30 origins
splits 1-15 (15), 30 origins
splits 1-15 (15), 30 origins
                   min  median     max
vs                                    
Basisformer     0.0050  0.0887  0.1595
Chain ensemble  0.0008  0.0363  0.1882
Dlinear         0.0038  0.0071  0.0235
GPT4TS          0.0000  0.0000  0.0000
MTSMixer        0.0000  0.0000  0.0000
PatchTST        0.5101  0.6358  0.7168
Sundial         0.2723  0.3729  0.5227
TSMixer         0.0000  0.0030  0.0231
TimeMoe         0.8408  0.8736  0.9145
TimesFM         0.6253  0.6830  0.7387
runs significant at 5%: {'Basisformer': 2, 'Chain ensemble': 6, 'Dlinear': 10, 'GPT4TS': 10, 'MTSMixer': 10, 'PatchTST': 0, 'Sundial': 0, 'TSMixer': 10, 'TimeMoe': 0, 'TimesFM': 0}


In [42]:
from statsmodels.stats.multitest import multipletests

P_adj = P.apply(
    lambda col: pd.Series(
        multipletests(col.values, alpha=0.05, method="fdr_bh")[1], index=col.index
    ),
    axis=0,
)

print(P_adj.agg(['min', 'median', 'max'], axis=1).round(4))
print('runs significant at 5% after BH:', (P_adj < 0.05).sum(axis=1).to_dict())

                   min  median     max
vs                                    
Basisformer     0.0125  0.1478  0.2658
Chain ensemble  0.0020  0.0726  0.3137
Dlinear         0.0076  0.0159  0.0578
GPT4TS          0.0000  0.0000  0.0000
MTSMixer        0.0000  0.0000  0.0000
PatchTST        0.6376  0.7589  0.8208
Sundial         0.3890  0.5327  0.7467
TSMixer         0.0000  0.0084  0.0578
TimeMoe         0.8408  0.8736  0.9145
TimesFM         0.6948  0.7589  0.8208
runs significant at 5% after BH: {'Basisformer': 1, 'Chain ensemble': 4, 'Dlinear': 8, 'GPT4TS': 10, 'MTSMixer': 10, 'PatchTST': 0, 'Sundial': 0, 'TSMixer': 9, 'TimeMoe': 0, 'TimesFM': 0}


### norm=1 (MAE)

In [3]:
SEQ_LEN = 48
STRIDE = SEQ_LEN
p_values = {}
for run in range(1,11,1):
    models = build_model_forcasts(run=run, seq_len=SEQ_LEN, chain_keep=[1, 3, 4, 5, 6, 8])
    fc = dm.stack(models=models, stride=STRIDE)
    p_values[run] = dm.dm_matrix(fc=fc, focus='Tree ensemble', norm=1).set_index('vs')['p_focus_better']

P = pd.DataFrame(p_values)
print(P.agg(['min', 'median', 'max'], axis=1).round(4))
print('runs significant at 5%:', (P < 0.05).sum(axis=1).to_dict())

splits 1-15 (15), 105 origins
splits 1-15 (15), 105 origins
splits 1-15 (15), 105 origins
splits 1-15 (15), 105 origins
splits 1-15 (15), 105 origins
splits 1-15 (15), 105 origins
splits 1-15 (15), 105 origins
splits 1-15 (15), 105 origins
splits 1-15 (15), 105 origins
splits 1-15 (15), 105 origins
                   min  median     max
vs                                    
Basisformer     0.0002  0.0009  0.0062
Chain ensemble  0.8545  0.9494  0.9786
Dlinear         0.0000  0.0000  0.0000
GPT4TS          0.0000  0.0000  0.0000
MTSMixer        0.0000  0.0000  0.0000
PatchTST        0.0104  0.0155  0.0428
Sundial         0.0000  0.0000  0.0004
TSMixer         0.0000  0.0000  0.0000
TimeMoe         0.0984  0.2072  0.2642
TimesFM         0.0001  0.0003  0.0011
runs significant at 5%: {'Basisformer': 10, 'Chain ensemble': 0, 'Dlinear': 10, 'GPT4TS': 10, 'MTSMixer': 10, 'PatchTST': 10, 'Sundial': 10, 'TSMixer': 10, 'TimeMoe': 0, 'TimesFM': 10}


In [4]:
from statsmodels.stats.multitest import multipletests

P_adj = P.apply(
    lambda col: pd.Series(
        multipletests(col.values, alpha=0.05, method="fdr_bh")[1], index=col.index
    ),
    axis=0,
)

print(P_adj.agg(['min', 'median', 'max'], axis=1).round(4))
print('runs significant at 5% after BH:', (P_adj < 0.05).sum(axis=1).to_dict())

                   min  median     max
vs                                    
Basisformer     0.0003  0.0013  0.0089
Chain ensemble  0.8545  0.9494  0.9786
Dlinear         0.0000  0.0000  0.0000
GPT4TS          0.0000  0.0000  0.0000
MTSMixer        0.0000  0.0000  0.0000
PatchTST        0.0130  0.0194  0.0535
Sundial         0.0000  0.0000  0.0008
TSMixer         0.0000  0.0000  0.0000
TimeMoe         0.1093  0.2303  0.2936
TimesFM         0.0002  0.0005  0.0018
runs significant at 5% after BH: {'Basisformer': 10, 'Chain ensemble': 0, 'Dlinear': 10, 'GPT4TS': 10, 'MTSMixer': 10, 'PatchTST': 9, 'Sundial': 10, 'TSMixer': 10, 'TimeMoe': 0, 'TimesFM': 10}


In [5]:
SEQ_LEN = 96
STRIDE = SEQ_LEN
p_values = {}
for run in range(1,11,1):
    models = build_model_forcasts(run=run, seq_len=SEQ_LEN, chain_keep=[1, 3, 4, 5, 6, 8])
    fc = dm.stack(models=models, stride=STRIDE)
    p_values[run] = dm.dm_matrix(fc=fc, focus='Tree ensemble', norm=1).set_index('vs')['p_focus_better']

P = pd.DataFrame(p_values)
print(P.agg(['min', 'median', 'max'], axis=1).round(4))
print('runs significant at 5%:', (P < 0.05).sum(axis=1).to_dict())

splits 1-15 (15), 60 origins
splits 1-15 (15), 60 origins
splits 1-15 (15), 60 origins
splits 1-15 (15), 60 origins
splits 1-15 (15), 60 origins
splits 1-15 (15), 60 origins
splits 1-15 (15), 60 origins
splits 1-15 (15), 60 origins
splits 1-15 (15), 60 origins
splits 1-15 (15), 60 origins
                   min  median     max
vs                                    
Basisformer     0.0000  0.0004  0.0264
Chain ensemble  0.3374  0.9115  0.9648
Dlinear         0.0000  0.0000  0.0000
GPT4TS          0.0000  0.0000  0.0000
MTSMixer        0.0000  0.0000  0.0000
PatchTST        0.0202  0.0532  0.1558
Sundial         0.0188  0.0524  0.1744
TSMixer         0.0000  0.0000  0.0006
TimeMoe         0.9180  0.9581  0.9892
TimesFM         0.1122  0.1887  0.3529
runs significant at 5%: {'Basisformer': 10, 'Chain ensemble': 0, 'Dlinear': 10, 'GPT4TS': 10, 'MTSMixer': 10, 'PatchTST': 5, 'Sundial': 5, 'TSMixer': 10, 'TimeMoe': 0, 'TimesFM': 0}


In [6]:

P_adj = P.apply(
    lambda col: pd.Series(
        multipletests(col.values, alpha=0.05, method="fdr_bh")[1], index=col.index
    ),
    axis=0,
)

print(P_adj.agg(['min', 'median', 'max'], axis=1).round(4))
print('runs significant at 5% after BH:', (P_adj < 0.05).sum(axis=1).to_dict())

                   min  median     max
vs                                    
Basisformer     0.0000  0.0009  0.0528
Chain ensemble  0.3749  0.9550  0.9892
Dlinear         0.0000  0.0000  0.0000
GPT4TS          0.0000  0.0000  0.0000
MTSMixer        0.0000  0.0000  0.0000
PatchTST        0.0289  0.0778  0.2491
Sundial         0.0289  0.0778  0.2491
TSMixer         0.0000  0.0000  0.0015
TimeMoe         0.9180  0.9621  0.9892
TimesFM         0.1402  0.2359  0.4411
runs significant at 5% after BH: {'Basisformer': 9, 'Chain ensemble': 0, 'Dlinear': 10, 'GPT4TS': 10, 'MTSMixer': 10, 'PatchTST': 2, 'Sundial': 2, 'TSMixer': 10, 'TimeMoe': 0, 'TimesFM': 0}


In [7]:
SEQ_LEN = 192
STRIDE = SEQ_LEN
p_values = {}
for run in range(1,11,1):
    models = build_model_forcasts(run=run, seq_len=SEQ_LEN, chain_keep=[1, 3, 4, 5, 8])
    fc = dm.stack(models=models, stride=STRIDE)
    p_values[run] = dm.dm_matrix(fc=fc, focus='Tree ensemble', norm=1).set_index('vs')['p_focus_better']

P = pd.DataFrame(p_values)
print(P.agg(['min', 'median', 'max'], axis=1).round(4))
print('runs significant at 5%:', (P < 0.05).sum(axis=1).to_dict())

splits 1-15 (15), 30 origins
splits 1-15 (15), 30 origins
splits 1-15 (15), 30 origins
splits 1-15 (15), 30 origins
splits 1-15 (15), 30 origins
splits 1-15 (15), 30 origins
splits 1-15 (15), 30 origins
splits 1-15 (15), 30 origins
splits 1-15 (15), 30 origins
splits 1-15 (15), 30 origins
                   min  median     max
vs                                    
Basisformer     0.0127  0.1186  0.2582
Chain ensemble  0.0001  0.0224  0.1509
Dlinear         0.0018  0.0048  0.0269
GPT4TS          0.0000  0.0000  0.0000
MTSMixer        0.0000  0.0000  0.0000
PatchTST        0.5750  0.6884  0.7962
Sundial         0.5434  0.6644  0.7620
TSMixer         0.0000  0.0003  0.0081
TimeMoe         0.8781  0.9218  0.9486
TimesFM         0.7449  0.8154  0.8631
runs significant at 5%: {'Basisformer': 1, 'Chain ensemble': 8, 'Dlinear': 10, 'GPT4TS': 10, 'MTSMixer': 10, 'PatchTST': 0, 'Sundial': 0, 'TSMixer': 10, 'TimeMoe': 0, 'TimesFM': 0}


In [8]:
P_adj = P.apply(
    lambda col: pd.Series(
        multipletests(col.values, alpha=0.05, method="fdr_bh")[1], index=col.index
    ),
    axis=0,
)

print(P_adj.agg(['min', 'median', 'max'], axis=1).round(4))
print('runs significant at 5% after BH:', (P_adj < 0.05).sum(axis=1).to_dict())

                   min  median     max
vs                                    
Basisformer     0.0254  0.1977  0.4303
Chain ensemble  0.0002  0.0482  0.2515
Dlinear         0.0045  0.0109  0.0538
GPT4TS          0.0000  0.0000  0.0000
MTSMixer        0.0000  0.0000  0.0000
PatchTST        0.7587  0.8605  0.9486
Sundial         0.7587  0.8605  0.9486
TSMixer         0.0000  0.0010  0.0270
TimeMoe         0.8781  0.9218  0.9486
TimesFM         0.8277  0.9060  0.9486
runs significant at 5% after BH: {'Basisformer': 1, 'Chain ensemble': 5, 'Dlinear': 9, 'GPT4TS': 10, 'MTSMixer': 10, 'PatchTST': 0, 'Sundial': 0, 'TSMixer': 10, 'TimeMoe': 0, 'TimesFM': 0}


### vs. Autogluon (Cannot be tested as they do not align on splits)